## Behavioral analysis of RPE on memory (delay condition)

In this project, we seek to understand how reward-prediction error information stacks upon perceptual memorability to modify memory performance. 

In this delay condition, participants perform two memory tests: one immediately after learning, and one 24 hours after.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from glob import glob
import sys 
sys.path.append('/hpc/users/qasims01/resources/ImgRatingSorter/')
from util import dprime
import seaborn as sns
import arviz as az
import pymc as pm 
import pickle

base_dir = '/sc/arion'
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/'

sys.path.append('/hpc/users/qasims01/resources/utils/RLModels_K99/scripts')
import param_recov_utils
import model_recov_utils
from model_utils import *
from RWModel import RWModel
from WSLSModel import WSLSModel
from BayesModel import BayesModel
from RWModel_Fictive1 import RWModel_Fictive1
from RWModel_Fictive2 import RWModel_Fictive2
from RWModel_Unsigned import RWModel_Unsigned
from EWAModel import EWA
from VKFModel import VKFModel

from inspect import getmembers, isfunction
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from sklearn.metrics import r2_score, roc_curve, auc

import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2, ttest_ind, pearsonr, spearmanr, t, norm, zscore, wilcoxon, ranksums, mannwhitneyu, ttest_1samp
from scipy.ndimage import gaussian_filter


sequential_colors = sns.color_palette("BrBG", 2)

base_dir = '/sc/arion'
save_dir_plots = f'{base_dir}/projects/guLab/Salman/MemoryBanditManuscript/Figures/Exp2'
save_dir_data = f'{base_dir}/projects/guLab/Salman/MemoryBanditManuscript/Data/Exp2'

In [ ]:
from HierarchicalRW import HierarchicalRWModel
from HierarchicalRW_Fictive1 import HierarchicalRWModel_Fictive1
from HierarchicalRW_Unsigned import HierarchicalRWModel_Unsigned
from HierarchicalBayes import HierarchicalBayesModel
from HierarchicalVKF import HierarchicalVKFModel
from HierarchicalRW_Signed import HierarchicalRWModel_Signed
from HierarchicalWSLS import HierarchicalWSLSModel

In [ ]:
# Load learn df for some hierarchical modeling. 
learn_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_RWF1.csv')
full_data_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_df_RWF1.csv')


In [ ]:
learn_df = learn_df[learn_df.good_dm_performance]

In [ ]:
learn_df.participant.unique().shape[0]

In [ ]:
n_subj = learn_df.participant.unique().shape[0]

In [ ]:
actions = learn_df.gamble.values.reshape(n_subj, 80).T
rewards = learn_df.reward.values.reshape(n_subj, 80).T

model = HierarchicalRWModel(n_subj=n_subj)
    
m = model.make_model(actions, rewards)

In [ ]:
f = pm.model_to_graphviz(m)
f.render(filename='/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/regression_model_viz',
             format='pdf');

In [ ]:
idata_true = az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_centered')

true_df = az.summary(idata_true)
true_alpha = true_df[true_df.index.str.startswith("alpha[")]['mean'].values
true_beta = true_df[true_df.index.str.startswith("beta[")]['mean'].values


In [ ]:
for ix, subj in enumerate(learn_df.participant.unique()):
    learn_df.loc[learn_df.participant==subj, 'alpha'] = true_alpha[ix]
    learn_df.loc[learn_df.participant==subj, 'beta'] = true_beta[ix]
    
    full_data_df.loc[full_data_df.participant==subj, 'alpha'] = true_alpha[ix]
    full_data_df.loc[full_data_df.participant==subj, 'beta'] = true_beta[ix]    
    
    a = learn_df.loc[learn_df.participant==subj, 'gamble'].values
    r = learn_df.loc[learn_df.participant==subj, 'reward'].values
    n_trials = len(a)
    # initialise Q values for each of the two options
    Qs = np.zeros((n_trials, 2))
    rpe = np.zeros(n_trials)

    # Initialize Q table
    Q = np.array([0.5, 0.5])
    
    for t in range(n_trials):
        # Update Q table
        rpe[t] = r[t] - Q[int(a[t])]
        Q[int(a[t])] = Q[int(a[t])] + true_alpha[ix] * (rpe[t])
        
    learn_df.loc[learn_df.participant==subj, 'rpe'] = rpe
    

In [ ]:
full_data_df.drop(columns=['rpe'],inplace=True)

In [ ]:
full_data_df = full_data_df.merge(learn_df[['participant', 'img_path', 'rpe']], on=['participant', 'img_path'], how='left')


In [ ]:
sns.regplot(data=full_data_df, x='rpe', y='hits', scatter=False, logistic=True)

In [ ]:
learn_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_HierarchicalRW.csv', index=False)
full_data_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_df_HierarchicalRW.csv', index=False)

In [ ]:
with pm.Model() as m:
    # Baribault parameterization: 
    a1 = pm.Gamma(name='a1', alpha=3, beta=1)
    a2 = pm.Gamma(name='a2', alpha=3, beta=1)
    b1 = pm.Gamma(name='b1', alpha=5, beta=1)
    mu_beta = pm.Normal(name='mu_beta', mu=7.5, sigma=2.5)
    b2 = pm.Deterministic('b2', (1+b1)/mu_beta)
    
    # centered 
    alpha = pm.Beta(name='alpha', alpha=1+a1, beta=1+a2, shape=200)
    beta = pm.Gamma(name='beta', alpha=1+b1, beta=b2, shape=200)   
    
draws = pm.draw([alpha, beta], draws=3, random_seed=1)
true_alpha = draws[0].ravel()
true_beta = draws[1].ravel()

In [ ]:
plt.hist(draws[1].ravel())
# plt.hist(draws[1].ravel())

In [ ]:
# sample from the priors (RW model) and simulate actions and rewards:

with pm.Model() as m:
    # Baribault parameterization: 
    a1 = pm.Gamma(name='a1', alpha=3, beta=1)
    a2 = pm.Gamma(name='a2', alpha=3, beta=1)
    b1 = pm.Gamma(name='b1', alpha=5, beta=1)
    mu_beta = pm.Normal(name='mu_beta', mu=7.5, sigma=2.5)
    b2 = pm.Deterministic('b2', (1+b1)/mu_beta)
    
    # centered 
    alpha = pm.Beta(name='alpha', alpha=1+a1, beta=1+a2, shape=200)
    beta = pm.Gamma(name='beta', alpha=1+b1, beta=b2, shape=200)   
    
draws = pm.draw([alpha, beta], draws=3, random_seed=1)
true_alpha = draws[0].ravel()
true_beta = draws[1].ravel()
    
draws = pm.draw([alpha, beta], draws=1, random_seed=1)
true_alpha = draws[0]
true_beta = draws[1]

n_trials = 80
n_subj = true_alpha.shape[0]

mus = [[.813, .187], [.177, .823], [.93, 0.07], [0.35, 0.65], [0.73, 0.27]]
actions = np.zeros([n_trials, n_subj], dtype="int")
rewards = np.zeros([n_trials, n_subj], dtype="int")

for subj in range(n_subj):
    
    # initialise Q values for each of the two options
    Qs = np.zeros((n_trials, 2))
    rpe = np.zeros(n_trials)

    # Initialize Q table
    Q = np.array([0.5, 0.5])
    reversal_trials = [16, 33, 48, 65]
    block = 0
    for t in range(n_trials):
        if t in reversal_trials: 
            block += 1

        # Apply the Softmax transformation
        exp_Q = np.exp(true_beta[subj] * Q)
        prob_a = exp_Q / np.sum(exp_Q)

        # Simulate choice and reward
        a = np.random.choice([0, 1], p=prob_a)
        r = np.random.random() < mus[block][a]

        rpe = r - Q[a]
        # Update Q table
        Q[a] = Q[a] + true_alpha[subj] * rpe

        # Store values
        actions[t, subj] = a
        rewards[t, subj] = r
        Qs[t] = Q.copy()
        
# Create a model with the fake data: 

model = HierarchicalRWModel(n_subj=n_subj)
    
m = model.make_model(actions, rewards)

# Fit the model: 

idata = model.fit(m, tune=1000, target_accept=0.8)

# Extract the parameters: 

sim_df = az.summary(idata)

sim_alpha = sim_df[sim_df.index.str.startswith("alpha[")]['mean']
sim_alpha_3 = sim_df[sim_df.index.str.startswith("alpha[")]['hdi_3%']
sim_alpha_97 = sim_df[sim_df.index.str.startswith("alpha[")]['hdi_97%']

sim_beta = sim_df[sim_df.index.str.startswith("beta[")]['mean']
sim_beta_3 = sim_df[sim_df.index.str.startswith("beta[")]['hdi_3%']
sim_beta_97 = sim_df[sim_df.index.str.startswith("beta[")]['hdi_97%']



In [ ]:
# Create the scatter plot
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/parameter_recovery_alpha.pdf') as pdf:

    f = plt.figure(figsize=(8, 8))
    plt.errorbar(true_alpha, sim_alpha, yerr=[sim_alpha - sim_alpha_3, 
                                         sim_alpha_97 - sim_alpha], 
                 fmt='o', markersize=10, capsize=3, elinewidth=1, capthick=1, alpha=0.5)
    plt.plot([0,1], [0,1], 'k--')
    plt.xlim([0,1])
    plt.ylim([0,1])
    
    pdf.savefig()

with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/parameter_recovery_beta.pdf') as pdf:

    # Create the scatter plot
    f = plt.figure(figsize=(8, 8))
    plt.errorbar(true_beta, sim_beta, yerr=[sim_beta - sim_beta_3, 
                                         sim_beta_97 - sim_beta], 
                 fmt='o', markersize=10, capsize=3, elinewidth=1, capthick=1, alpha=0.5)
    plt.plot([0, 15], [0, 15], 'k--')
    plt.xlim([0, 15])
    plt.ylim([0, 15])
    
    pdf.savefig()

In [ ]:
pearsonr(true_alpha, sim_alpha.values)

In [ ]:
pearsonr(true_beta, sim_beta.values)

In [ ]:
idata_true = az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Signed_centered')

true_df = az.summary(idata_true)
true_alpha_pos = true_df[true_df.index.str.startswith("alpha_pos[")]['mean'].values
true_alpha_neg = true_df[true_df.index.str.startswith("alpha_neg[")]['mean'].values
true_beta = true_df[true_df.index.str.startswith("beta[")]['mean'].values


In [ ]:
# sample from the priors (RWS model) and simulate actions and rewards:

# with pm.Model() as m:

#     # Baribault parameterization: 

#     b1 = pm.Gamma(name='b1', alpha=5, beta=1)
#     mu_beta = pm.Normal(name='mu_beta', mu=7.5, sigma=2.5)
#     b2 = pm.Deterministic('b2', (1+b1)/mu_beta)

#     # Centered
#     a1_pos = pm.Gamma(name='a1_pos', alpha=2, beta=1)
#     a2_pos = pm.Gamma(name='a2_pos', alpha=5, beta=1)

#     a1_neg = pm.Gamma(name='a1_neg', alpha=2, beta=1)
#     a2_neg = pm.Gamma(name='a2_neg', alpha=5, beta=1)

#     alpha_pos = pm.Beta(name='alpha_pos', alpha=1+a1_pos, beta=1+a2_pos, shape=self.n_subj)
#     alpha_neg = pm.Beta(name='alpha_neg', alpha=1+a1_neg, beta=1+a2_neg, shape=self.n_subj)
#     beta = pm.Gamma(name='beta', alpha=1+b1, beta=b2, shape=self.n_subj)
    
# draws = pm.draw([alpha_neg, alpha_pos, beta], draws=1, random_seed=1)
# true_alpha_neg = draws[0]
# true_alpha_pos = draws[1]
# true_beta = draws[2]

n_trials = 80
n_subj = true_alpha_neg.shape[0]

mus = [[.813, .187], [.177, .823], [.93, 0.07], [0.35, 0.65], [0.73, 0.27]]
actions = np.zeros([n_trials, n_subj], dtype="int")
rewards = np.zeros([n_trials, n_subj], dtype="int")

for subj in range(n_subj):
    
    # initialise Q values for each of the two options
    Qs = np.zeros((n_trials, 2))
    rpe = np.zeros(n_trials)

    # Initialize Q table
    Q = np.array([0.5, 0.5])
    reversal_trials = [16, 33, 48, 65]
    block = 0
    for t in range(n_trials):
        if t in reversal_trials: 
            block += 1

        # Apply the Softmax transformation
        exp_Q = np.exp(true_beta[subj] * Q)
        prob_a = exp_Q / np.sum(exp_Q)

        # Simulate choice and reward
        a = np.random.choice([0, 1], p=prob_a)
        r = np.random.random() < mus[block][a]

        rpe = (1 - Q[a]) * r
        nrpe = (0 - Q[a]) * (1 - r)

        # Update Q table
        Q[a] = Q[a] + true_alpha_pos[subj] * rpe + true_alpha_neg[subj] * nrpe

        # Store values
        actions[t, subj] = a
        rewards[t, subj] = r
        Qs[t] = Q.copy()
        
# Create a model with the fake data: 

model = HierarchicalRWModel_Signed(n_subj=n_subj)
    
m = model.make_model(actions, rewards)

# Fit the model: 

idata = model.fit(m, tune=1000, target_accept=0.8)

# Extract the parameters: 

sim_df = az.summary(idata)

sim_alpha_neg = sim_df[sim_df.index.str.startswith("alpha_neg[")]['mean']
sim_alpha_neg_3 = sim_df[sim_df.index.str.startswith("alpha_neg[")]['hdi_3%']
sim_alpha_neg_97 = sim_df[sim_df.index.str.startswith("alpha_neg[")]['hdi_97%']

sim_alpha_pos = sim_df[sim_df.index.str.startswith("alpha_pos[")]['mean']
sim_alpha_pos_3 = sim_df[sim_df.index.str.startswith("alpha_pos[")]['hdi_3%']
sim_alpha_pos_97 = sim_df[sim_df.index.str.startswith("alpha_pos[")]['hdi_97%']

sim_beta = sim_df[sim_df.index.str.startswith("beta[")]['mean']
sim_beta_3 = sim_df[sim_df.index.str.startswith("beta[")]['hdi_3%']
sim_beta_97 = sim_df[sim_df.index.str.startswith("beta[")]['hdi_97%']


In [ ]:
# Create the scatter plot
plt.figure(figsize=(8, 8))
plt.errorbar(true_alpha_neg, sim_alpha_neg, yerr=[sim_alpha_neg - sim_alpha_neg_3, 
                                     sim_alpha_neg_97 - sim_alpha_neg], 
             fmt='o', markersize=4, capsize=3, elinewidth=1, capthick=1)
plt.plot([0.5,1.15], [0.5,1.15], 'k--')
plt.xlim([0.5,1.15])
plt.ylim([0.5,1.15])

# Create the scatter plot
plt.figure(figsize=(8, 8))
plt.errorbar(true_alpha_pos, sim_alpha_pos, yerr=[sim_alpha_pos - sim_alpha_pos_3, 
                                     sim_alpha_pos_97 - sim_alpha_pos], 
             fmt='o', markersize=4, capsize=3, elinewidth=1, capthick=1)
plt.plot([0.5,1.15], [0.5,1.15], 'k--')
plt.xlim([0.5,1.15])
plt.ylim([0.5,1.15])

# Create the scatter plot
plt.figure(figsize=(8, 8))
plt.errorbar(true_beta, sim_beta, yerr=[sim_beta - sim_beta_3, 
                                     sim_beta_97 - sim_beta], 
             fmt='o', markersize=4, capsize=3, elinewidth=1, capthick=1)
plt.plot([0, 10], [0, 10], 'k--')
plt.xlim([0, 10])
plt.ylim([0, 10])

In [ ]:
pearsonr(true_alpha_pos, sim_alpha_pos.values)

In [ ]:
pearsonr(true_alpha_neg, sim_alpha_neg.values)

In [ ]:
pearsonr(true_beta, sim_beta.values)

In [ ]:
idata = model.fit(m, tune=1000, target_accept=0.8)

# with open(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Unsigned.pickle', 'wb') as handle:
#     pickle.dump(idata2, handle,
#            protocol=pickle.HIGHEST_PROTOCOL)
#     print("Saved everything!")

In [ ]:
# Calculate means and standard errors
means = sim_df[4:204]['mean']


# Create the scatter plot
plt.figure(figsize=(8, 8))
plt.errorbar(true_alpha, means, yerr=[means - sim_df[4:204]['hdi_3%'], 
                                     sim_df[4:204]['hdi_97%'] - means], 
             fmt='o', markersize=4, capsize=3, elinewidth=1, capthick=1)
plt.plot([0,0.85], [0,0.85], 'k--')
plt.xlim([0,0.85])
plt.ylim([0,0.85])

In [ ]:
pearsonr(true_alpha, means)

In [ ]:
# Calculate means and standard errors
means = sim_df[-201:-1]['mean']

# Create the scatter plot
plt.figure(figsize=(8, 8))
plt.errorbar(true_beta, means, yerr=[means - sim_df[-201:-1]['hdi_3%'], 
                                     sim_df[-201:-1]['hdi_97%'] - means], 
             fmt='o', markersize=4, capsize=3, elinewidth=1, capthick=1)
plt.plot([0,6.5], [0,6.5], 'k--')
plt.xlim([0,6.5])
plt.ylim([0,6.5])

In [ ]:
pearsonr(true_beta, means)

In [ ]:
az.to_netcdf(idata, f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_centered')


In [ ]:
idata = model.fit(m, tune=1000, target_accept=0.9)


In [ ]:
df_comp = az.compare({'RW':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_centered'),
                      'WSLS': az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalWSLSModel'),
                     'RWUS':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Unsigned'), 
                     'RWF':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Fictive1'),
                     'Bayes':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalBayesModel'),
                     'VKF': az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalVKFModel'),
                     'RWS':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Signed_centered')},
                     ic='waic')

In [ ]:
with PdfPages(f'{save_dir_plots}/HierarchicalModelComparison.pdf') as pdf:

    az.plot_compare(df_comp, figsize=[4,4], plot_ic_diff=False)
    
    pdf.savefig()

In [ ]:
save_dir_plots

In [ ]:
az.plot_compare?

In [ ]:
idata2 = az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Signed')

In [ ]:
az.plot_trace(idata)


In [ ]:
az.plot_trace(idata2)


In [ ]:
az.waic(idata)

In [ ]:
mod_df = az.summary(idata)

In [ ]:
mod_df2[-401:-201]

In [ ]:
mod_df2 = az.summary(idata2)

In [ ]:
mod_df2[-601:-401]

In [ ]:
sns.regplot(y=mod_df2[-401:-201]['mean'], x=mod_df2[-601:-401]['mean'])

In [ ]:
sns.regplot(y=mod_df2[-401:-201]['mean'], x=mod_df.iloc[4:204]['mean'])

In [ ]:
plt.hist(mod_df2[-602:-402]['mean'])

In [ ]:
plt.hist(mod_df2[-400:-200]['mean'])

In [ ]:
az.loo(idata)

In [ ]:
sns.regplot(x=learn_df.groupby('participant').mean().reset_index()['beta'], 
            y=mod_df2[-200:]['mean'].values)

In [ ]:
sns.regplot(x=learn_df.groupby('participant').mean().reset_index()['beta'], 
            y=mod_df2[-201:-1]['mean'].values)

In [ ]:
sns.regplot(x=mod_df2[4:204]['mean'].values, 
           y=mod_df2[-201:-1]['mean'].values)

In [ ]:
ax = sns.jointplot(x=mod_df2[-400:-200]['mean'].values, 
                   y=mod_df2[-200:]['mean'].values, 
                   height=8, ratio=2, kind='kde', shade=True, cmap='Spectral_r', 
                   fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
# ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
# ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)

In [ ]:
alpha1 = mod_df2[-400:-200]['mean'].values

In [ ]:
alpha2 = learn_df.iloc[0:80].alpha.values[0]

In [ ]:
plt.hist(rpe1)

In [ ]:
n_trials=80
# initialise Q values for each of the two options
Q = np.array([0.5, 0.5])
rpe1 = np.zeros(n_trials)
Qs = [] 
for t in range(n_trials):
    # Update Q table
    rpe1[t] = rewards[t, 0] - Q[int(actions[t, 0])]
    Q[int(actions[t, 0])] = Q[int(actions[t, 0])] + alpha1[0] * (rpe1[t])
    Qs.append(Q[1])


In [ ]:
plt.hist(rpe1)

In [ ]:
plt.plot(Qs)

In [ ]:
sns.regplot(x=rpe1, y=rpe2)
# sns.regplot(x=learn_df.iloc[0:80].reward, y=learn_df.iloc[0:80].rpe)

In [ ]:
seaborn_colors = ['Accent', 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'BrBG_r', 'BuGn', 'BuGn_r', 'BuPu', 'BuPu_r', 
 'CMRmap', 'CMRmap_r', 'Dark2', 'Dark2_r', 'GnBu', 'GnBu_r', 'Greens', 'Greens_r', 'Greys', 'Greys_r', 'OrRd', 
 'OrRd_r', 'Oranges', 'Oranges_r', 'PRGn', 'PRGn_r', 'Paired', 'Paired_r', 'Pastel1', 
 'Pastel1_r', 'Pastel2', 'Pastel2_r', 'PiYG', 'PiYG_r', 'PuBu', 'PuBuGn', 'PuBuGn_r', 
 'PuBu_r', 'PuOr', 'PuOr_r', 'PuRd', 'PuRd_r', 'Purples', 'Purples_r', 'RdBu', 'RdBu_r', 
 'RdGy', 'RdGy_r', 'RdPu', 'RdPu_r', 'RdYlBu', 'RdYlBu_r', 'RdYlGn', 'RdYlGn_r', 'Reds', 
 'Reds_r', 'Set1', 'Set1_r', 'Set2', 'Set2_r', 'Set3', 'Set3_r', 'Spectral', 'Spectral_r', 
 'Wistia', 'Wistia_r', 'YlGn', 'YlGnBu', 'YlGnBu_r', 'YlGn_r', 'YlOrBr', 'YlOrBr_r', 'YlOrRd', 
 'YlOrRd_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'binary', 'binary_r', 'bone', 
 'bone_r', 'brg', 'brg_r', 'bwr', 'bwr_r', 'cividis', 'cividis_r', 'cool', 'cool_r', 'coolwarm', 'coolwarm_r', 'copper', 'copper_r',
 'cubehelix', 'cubehelix_r', 'flag', 'flag_r', 'gist_earth', 'gist_earth_r', 'gist_gray', 'gist_gray_r', 'gist_heat', 'gist_heat_r', 'gist_ncar', 'gist_ncar_r',
 'gist_rainbow', 'gist_rainbow_r', 'gist_stern', 'gist_stern_r', 'gist_yarg', 
 'gist_yarg_r', 'gnuplot', 'gnuplot2', 'gnuplot2_r', 'gnuplot_r', 'gray', 'gray_r',
 'hot', 'hot_r', 'hsv', 'hsv_r', 'icefire', 'icefire_r', 'inferno', 
 'inferno_r', 'magma', 'magma_r', 'mako', 'mako_r', 
 'nipy_spectral', 'nipy_spectral_r', 'ocean', 'ocean_r', 'pink', 'pink_r',
 'plasma', 'plasma_r', 'prism', 'prism_r', 'rainbow', 'rainbow_r',
 'rocket', 'rocket_r', 'seismic', 'seismic_r', 'spring', 'spring_r',
 'summer', 'summer_r', 'tab10', 'tab10_r', 'tab20', 'tab20_r', 'tab20b',
 'tab20b_r', 'tab20c', 'tab20c_r', 'terrain', 'terrain_r', 'twilight',
 'twilight_r', 'twilight_shifted', 'twilight_shifted_r', 'viridis', 'viridis_r', 'vlag', 'vlag_r', 'winter', 'winter_r']



In [ ]:
sns.set_context("paper")


In [ ]:
import time, sys
from IPython.display import clear_output

def update_progress(progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = "Progress: [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

In [ ]:
def compare_corr(xy, ab, n, n2 = None, twotailed=True):
    """
    Calculates the statistic significance between two independent correlation coefficients
    @param xy: correlation coefficient between x and y
    @param xz: correlation coefficient between a and b
    @param n: number of elements in xy
    @param n2: number of elements in ab (if distinct from n)
    @param twotailed: whether to calculate a one or two tailed test, only works for 'fisher' method
    @param conf_level: confidence level, only works for 'zou' method
    @param method: defines the method uses, 'fisher' or 'zou'
    @return: z and p-val
    """

    xy_z = 0.5 * np.log((1 + xy)/(1 - xy))
    ab_z = 0.5 * np.log((1 + ab)/(1 - ab))
    if n2 is None:
        n2 = n

    se_diff_r = np.sqrt(1/(n - 3) + 1/(n2 - 3))
    diff = xy_z - ab_z
    z = abs(diff / se_diff_r)
    p = (1 - norm.cdf(z))
    if twotailed:
        p *= 2

    return z, p

In [ ]:
def matthews_corrcoef(tp, tn, fp, fn):
    """
    """
    
    numerator = (tp * tn) - (fp * fn)
    
    denominator = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
    
    if denominator == 0:
        return 0
    
    return numerator / denominator

In [ ]:
import arviz as az
import bambi as bmb
import argparse
from os.path import join
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt 

# import pymc3 as pm
# import pymc3.sampling_jax
# import theano

def run_model(df, y=None, X=None, Intx=None, rand_effect=None, rand_slopes=False, within_subject_X=None,
              within_subject_Intx=None, priors=None, categorical=None,
              cores=2, chains=2, tune=1500, draws=2000, target_accept=0.95, model_fam='bernoulli', 
              output_dir='/home1/salman.qasim/Salman_Project/FR_Emotion/BayesModels', return_model=False,
              categorical_baseline=True, save_model_res = True, label=None):
    """
    Run the model: 
    
    Parameters
    ----------
    df: pandas dataframe 
        All the variables 
    y: str
        Name of the dependent variable
    X: list of strings 
        Names of the independent variables 
    Intx: list of strings
        Names of the interactions, formatted as 'a:b' 
    rand_effect: str
        Name of variable for random effects (i.e. 'subject') 
    rand_slopes: bool 
        Should we fit random slopes in addition to random intercepts? 
    priors: dict 
        Prior distribution for any independent variables (i.e. Intercept, random intercept) 
        Default to weakly informative priors (Gelman et al. 2008) 
    categorical: list of strings 
        Names of categorical variables going into model 
    
    Returns
    -------
    results: model
    
    
    y = 'memory_outcome'
    X = ['sds_score', 'zrpe', 'zpm', 'gamble_rt_log', 'recog_rt_log', 'img_cat', 'age', 'Sex', 'alpha_pos', 'alpha_neg', 'beta']
    Intx = ['sds_score:zrpe', 'sds_score:zpm']
    rand_effect = ['participant']
    categorical = ['img_cat', 'Sex']

    # Drop nan data
    model_df = model_df.dropna(subset=X)

    model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=False, 
                               priors=None, categorical=categorical, cores=4, chains=4, tune=1500, draws=2000, 
                               target_accept=0.9, model_fam='bernoulli', 
                               output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                               return_model=True,
                               categorical_baseline=True, 
                               save_model_res = True, 
                               label=None)
    
    """

    
    # if no label provided, make your own
    if not label: 
        label = (f"{y}" + "_{}"*len(X)).format(*X)
        
    
    if categorical_baseline: # this means we set a baseline for categorical effects 
        # format random effects properly 
        rand_term = [f'(1|{x})' for x in rand_effect]
        if rand_slopes: # do we want a more complex mixed-effects model with random slopes as well as intercepts?
            if not within_subject_X:
                raise ValueError('No within subject variable specified')
            if not Intx:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])
            elif not within_subject_Intx:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])+'+'+'+'.join(Intx)
            else:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])+'+'+'+'.join(Intx)+'+'+'+'.join([f'({intx}|{rand_effect[0]})' for intx in within_subject_Intx])
        else:
            if not Intx: 
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)
            else:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)
    else: # this means all levels of categorical effect are judged absent or present 
         # format random effects properly 
        rand_term = [f'(0+1|{x})' for x in rand_effect]
        if rand_slopes: 
            formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'(0 + {x}|{rand_effect[0]})' for x in X])+'+'+'+'.join(Intx)+'+'+'+'.join([f'(0 + {intx}|{rand_effect[0]})' for intx in Intx])
        else:
            if not Intx:
                formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)
            else:
                formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)        

    print(formula)
    # construct the model 
    model = bmb.Model(formula=formula, 
                  data=df[rand_effect + [y] + X],
                 family=model_fam,
                 priors=priors,
                 categorical=categorical)
    
    # Future steps for speedup:
    
#     if jax: 
#         model.build() # before fitting using bambi get pymc backend model 
#         results =  model.backend.model.sampling_jax.sample_numpyro_nuts(draws, tune=tune, target_accept=target_accept)
#     else:

    # fit the model 
    results=model.fit(cores=cores, 
                          chains=chains, 
                          tune=tune, 
                          draws=draws, 
                          target_accept=target_accept)
    # init='advi+adapt_diag'

    # save out the CSV with the results 
    summary = az.summary(results, hdi_prob=0.95)
    for var in summary.index.values:
        if var.endswith("]"):  # skip individual subject parameters
            continue
        if var.startswith("1"):  # skip random effects.
            continue
        summary.loc[var, "P>0"] = np.mean(results.posterior[var].values > 0)
    summary.to_csv(join(output_dir , f"{label}_summary.csv"))

    # save out the HDI plot of all the fixed effects 
    axes = az.plot_forest(results,
                           kind='ridgeplot',
                           var_names=[f'^{x}' for x in X],
                           filter_vars="regex",
                           colors='black',
                           combined=True,
                           hdi_prob=0.95,
                           figsize=(9, 7))
    plt.vlines(0, plt.ylim()[0], plt.ylim()[1], color = 'black')
    # forestplot
#         plt.xlim([-0.2, 0.1])

    plt.savefig(join(output_dir, f'{label}_HDIplot.pdf'), dpi=300)
    plt.close()

    # print out the latex table for the cell output
    results_df = summary.loc[:,
                   ['mean', 'sd', 'hdi_2.5%', 'hdi_97.5%']
                  ]
    all_tests = [] 

    for x in X: 
        temp = results_df.filter(like=f'{x}', axis=0)
        all_tests.append(temp)

    results_df = pd.concat(all_tests)

    c_string = '|c'*results_df.shape[0] + '|'
#     print(results_df.reset_index().to_latex(index=False, 
#                                             column_format=c_string).replace("\\\n", "\\ \hline\n"))  

    if save_model_res:
        # save out the model for model comparison (THIS WILL BE GB-sized, so really only needed for model comparison)
        az.to_netcdf(results, join(output_dir , f"{label}_model"))
    
    if return_model:
        return model, results 

In [ ]:
def plot_res(fitted_model, X, ax=None, save_label=None): 
    """
    Code for plotting select beta weights in a publication friendly manner from saved csv
    """
        
    axes = az.plot_forest(fitted_model,
                           kind='ridgeplot',
                           var_names=[f'^(?!.*\|).*{x}.*$' for x in X],
                           filter_vars="regex",
                           colors='gray',
                           combined=True,
                           ridgeplot_overlap=2,
                           ridgeplot_truncate=False,
                           ridgeplot_alpha=0.4,
                           hdi_prob=0.95,
                           figsize=(9, 7),
                           ax=ax)
    # 'forestplot'
    plt.vlines(0, plt.ylim()[0], plt.ylim()[1], color = 'black')
    plt.show()
#     plt.savefig(save_label, dpi=300)
#     plt.close()

In [ ]:
def balance(seq):
    """
    
    Compute data set class balance using Shannon entropy divided by log(k) for k classes 
    
    """
    from collections import Counter
    from numpy import log
    
    n = len(seq)
    classes = [(clas,float(count)) for clas,count in Counter(seq).items()]
    k = len(classes)
    
    H = -sum([ (count/n) * log((count/n)) for clas,count in classes]) #shannon entropy
    return H/log(k)

# Data pre-processing

This should only really need to be done once unless introducing more data

In [ ]:
# Initial pilot data path 
load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/DelayedMemoryTest'

# Load the demographics data to get 
demo_file = f'{load_dir}/demographics.csv'
demographics_df = pd.read_csv(demo_file, index_col=False, header=0)
# Some patients demos are shifted oddly
demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Age'] = demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Sex']
demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Sex'] = demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Ethnicity simplified']
demographics_df = demographics_df[['Participant id', 'Age', 'Sex']]

demographics_df.rename(columns={'Participant id': 'participant',
                               'Age': 'age', 
                               'Sex': 'sex'}, inplace=True)

# Load the database for Day 1: 
db_file = glob(f'{load_dir}/Day1/db/*.csv')[0]
day1_df = pd.read_csv(db_file, index_col=None, header=0)
# Filter this to the people in the Prolific demographic data 
day1_df = day1_df[day1_df.participant.isin(demographics_df.participant.unique().tolist())].reset_index(drop=True)


# Load the database for Day 2: 
db_file = glob(f'{load_dir}/Day2/db/*.csv')[0]
day2_df = pd.read_csv(db_file, index_col=None, header=0)
# Filter this to the people in the Prolific demographic data 
day2_df = day2_df[day2_df.participant.isin(demographics_df.participant.unique().tolist())].reset_index(drop=True)

# Load the database of all the image d': 
all_mem_file = f'{base_dir}/projects/guLab/Salman/EMU/all_mem_data_long.xlsx' 
all_mem_df = pd.read_excel(all_mem_file, engine='openpyxl')
all_mem_df = all_mem_df[['img_path', 'DPRIME']]
all_mem_df.img_path = all_mem_df.img_path.apply(lambda x: x[:-4]+'.png' )

In [ ]:
# For my RPPR: 

demographics_df

In [ ]:
learn_df = day1_df[['participant', 'trials_2.thisRepN', 'block', 'prob', 'MB1_draw_key.keys', 'MB1_draw_key.rt', 'reward', 'img_path', 'Gender', 'trials_3.thisRepN']]
mem_df = day1_df[['participant', 'trials_3.thisRepN', 'MEM2_recall_key.keys', 'MEM2_recall_key.rt', 'MEM2_conf_slider.rating', 'MEM2_conf_slider.rt', 'trialIm']]

# clean up the learning database
two_ab_ntrials = 80
learn_df = learn_df[np.isnan(learn_df['trials_3.thisRepN'])].reset_index(drop=True)
trials = (learn_df[~np.isnan(learn_df['trials_2.thisRepN'])]['trials_2.thisRepN'].astype(int)+1).tolist()
learn_df = learn_df[~np.isnan(learn_df['reward'])].reset_index(drop=True)
learn_df = learn_df[np.isnan(learn_df['trials_2.thisRepN'])].dropna(subset=['Gender']).reset_index(drop=True)
learn_df['learn_trials'] = trials
learn_df.drop(columns=['trials_2.thisRepN', 'trials_3.thisRepN'], inplace=True)
learn_df.rename(columns={'MB1_draw_key.keys':'gamble', 'MB1_draw_key.rt': 'gamble_rt'}, inplace=True) 
# Code reward as 1 or 0
learn_df.reward = learn_df.reward/100
# Code gamble as 1 or 0 (0 is male, 1 is female)
learn_df.gamble = learn_df.gamble.apply(lambda x: 0 if x==1.0 else 1)
learn_df['male'] = 0
learn_df['female'] = 0
learn_df.male = learn_df.apply(lambda x: 1 if x.gamble==0 else 0, axis=1)
learn_df.female = learn_df.apply(lambda x: 1 if x.gamble==1 else 0, axis=1)


# clean up the memory database
mem_df = mem_df[~np.isnan(mem_df['trials_3.thisRepN'])].reset_index(drop=True)
mem_df.rename(columns={'trials_3.thisRepN': 'mem_trials',
                       'trialIm':'img_path', 
                       'MEM2_recall_key.keys': 'recog_choice',
                       'MEM2_recall_key.rt': 'recog_rt',
                       'MEM2_conf_slider.rating': 'conf_choice',
                       'MEM2_conf_slider.rt': 'conf_rt'}, inplace=True)
mem_df['condition'] = 'day_1'

# merge the two
data_df = mem_df.merge(learn_df, on=['participant', 'img_path'], how='left', indicator=True, suffixes=(None, '_learn'))

data_df['recog_choice_bool'] = data_df.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
data_df.recog_choice = data_df.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

data_df['hits'] = (data_df._merge=='both') & (data_df.recog_choice_bool==1)
data_df['misses'] = (data_df._merge=='both') & (data_df.recog_choice_bool==0)
data_df['false_alarms'] = (data_df._merge=='left_only') & (data_df.recog_choice_bool==1)
data_df['correct_rejections'] = (data_df._merge=='left_only') & (data_df.recog_choice_bool==0)

for pid in data_df.participant.unique():
    subj_df = data_df[data_df.participant==pid]

    # compute the hit-rates and false-alarm rates 
    hits = subj_df[(subj_df._merge=='both') & (subj_df.recog_choice_bool==1)].shape[0]
    misses = subj_df[(subj_df._merge=='both') & (subj_df.recog_choice_bool==0)].shape[0]

    false_alarms = subj_df[(subj_df._merge=='left_only') & (subj_df.recog_choice_bool==1)].shape[0]
    correct_rejections = subj_df[(subj_df._merge=='left_only') & (subj_df.recog_choice_bool==0)].shape[0]

    hm = hits+misses
    fc = false_alarms+correct_rejections
    hrs = hits/hm
    fars = false_alarms/fc

    dp, _, c = dprime(hrs, fars, hm, fc, criteria=True)
    
    mcc = matthews_corrcoef(hits, correct_rejections, false_alarms, misses)

    data_df.loc[data_df.participant==pid, 'hit_rate'] = hrs
    data_df.loc[data_df.participant==pid, 'false_alarm_rate'] = fars
    data_df.loc[data_df.participant==pid, 'dprime'] = np.nan
    data_df.loc[data_df.participant==pid, 'crit'] = np.nan
    data_df.loc[data_df.participant==pid, 'auc'] = np.nan
    
    # Compute AUC
    signal = subj_df[(subj_df.hits==1) | ((subj_df.misses==1)) ].recog_choice_bool
    noise = subj_df[(subj_df.correct_rejections==1) | ((subj_df.false_alarms==1)) ].recog_choice_bool
    n_samples = len(signal)
    responses = np.concatenate((signal, noise))  # Combine signal and noise responses
    true_labels = np.concatenate((np.ones(n_samples), np.zeros(n_samples)))  # 1 for signal, 0 for noise

    # Compute ROC curve and AUC score
    fpr, tpr, _ = roc_curve(true_labels, responses)
    roc_auc = auc(fpr, tpr)

    if dp != float("-inf"):
        data_df.loc[data_df.participant==pid, 'dprime'] = dp   
        data_df.loc[data_df.participant==pid, 'crit'] = c
        data_df.loc[data_df.participant==pid, 'mcc'] = mcc
        data_df.loc[data_df.participant==pid, 'auc'] = roc_auc


# Merge in the image DPRIME 
data_df = data_df.merge(all_mem_df, on='img_path', how='left', indicator=False)
data_df.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 


In [ ]:
day2_df[day2_df.participant=='607778a26ff3de83125f50c5'].trialIm.dropna().values

In [ ]:
# day2_df
mem_df2 = day2_df[['participant', 'trials.thisRepN', 'key_resp.keys', 'key_resp.rt', 
         'new_slider.rt', 'new_slider.rating', 'trialIm']].rename(columns={'trialIm':'img_path',
                                                                      'trials.thisRepN':'mem_trials',
                                                                      'key_resp.keys': 'recog_choice',
                                                                      'key_resp.rt': 'recog_rt',
                                                                      'new_slider.rating': 'conf_choice',
                                                                      'new_slider.rt': 'conf_rt'}).dropna(subset=['img_path'])
mem_df2['condition'] = 'day_2'

In [ ]:
mem_df2[mem_df2.participant=='6290c65ed46e5dce4f16bb92']

In [ ]:
# day2_df
mem_df2 = day2_df[['participant', 'trials.thisRepN', 'key_resp.keys', 'key_resp.rt', 
         'new_slider.rt', 'new_slider.rating', 'trialIm']].rename(columns={'trialIm':'img_path',
                                                                      'trials.thisRepN':'mem_trials',
                                                                      'key_resp.keys': 'recog_choice',
                                                                      'key_resp.rt': 'recog_rt',
                                                                      'new_slider.rating': 'conf_choice',
                                                                      'new_slider.rt': 'conf_rt'}).dropna(subset=['img_path'])
mem_df2['condition'] = 'day_2'

# merge the two
data_df2 = mem_df2.merge(learn_df, on=['participant', 'img_path'], how='left', indicator=True, suffixes=(None, '_learn'))

data_df2['recog_choice_bool'] = data_df2.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
data_df2.recog_choice = data_df2.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

data_df2['hits'] = (data_df2._merge=='both') & (data_df2.recog_choice_bool==1)
data_df2['misses'] = (data_df2._merge=='both') & (data_df2.recog_choice_bool==0)
data_df2['false_alarms'] = (data_df2._merge=='left_only') & (data_df2.recog_choice_bool==1)
data_df2['correct_rejections'] = (data_df2._merge=='left_only') & (data_df2.recog_choice_bool==0)

for pid in data_df2.participant.unique():
    subj_df = data_df2[data_df2.participant==pid]

    # compute the hit-rates and false-alarm rates 
    hits = subj_df[(subj_df._merge=='both') & (subj_df.recog_choice_bool==1)].shape[0]
    misses = subj_df[(subj_df._merge=='both') & (subj_df.recog_choice_bool==0)].shape[0]

    false_alarms = subj_df[(subj_df._merge=='left_only') & (subj_df.recog_choice_bool==1)].shape[0]
    correct_rejections = subj_df[(subj_df._merge=='left_only') & (subj_df.recog_choice_bool==0)].shape[0]

    hm = hits+misses
    fc = false_alarms+correct_rejections
    hrs = hits/hm
    fars = false_alarms/fc

    dp, _, c = dprime(hrs, fars, hm, fc, criteria=True)
    mcc = matthews_corrcoef(hits, correct_rejections, false_alarms, misses)

    data_df2.loc[data_df2.participant==pid, 'hit_rate'] = hrs
    data_df2.loc[data_df2.participant==pid, 'false_alarm_rate'] = fars
    data_df2.loc[data_df2.participant==pid, 'dprime'] = np.nan
    data_df2.loc[data_df2.participant==pid, 'crit'] = np.nan
    data_df2.loc[data_df2.participant==pid, 'auc'] = np.nan
    
    # Compute AUC
    signal = subj_df[(subj_df.hits==1) | ((subj_df.misses==1)) ].recog_choice_bool
    noise = subj_df[(subj_df.correct_rejections==1) | ((subj_df.false_alarms==1)) ].recog_choice_bool
    n_samples = len(signal)
    responses = np.concatenate((signal, noise))  # Combine signal and noise responses
    true_labels = np.concatenate((np.ones(n_samples), np.zeros(n_samples)))  # 1 for signal, 0 for noise

    # Compute ROC curve and AUC score
    fpr, tpr, _ = roc_curve(true_labels, responses)
    roc_auc = auc(fpr, tpr)

    if dp != float("-inf"):
        data_df2.loc[data_df2.participant==pid, 'dprime'] = dp  
        data_df2.loc[data_df2.participant==pid, 'crit'] = c   
        data_df2.loc[data_df2.participant==pid, 'mcc'] = mcc
        data_df2.loc[data_df2.participant==pid, 'auc'] = roc_auc
    

# Merge in the image DPRIME 
data_df2 = data_df2.merge(all_mem_df, on='img_path', how='left', indicator=False)
data_df2.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 


In [ ]:
full_data_df = pd.concat([data_df, data_df2])


In [ ]:

for pid in full_data_df.participant.unique():
    subj_df = full_data_df[full_data_df.participant==pid]
    
    # Add a filter for people with bad performance
    full_data_df.loc[full_data_df.participant==pid, 'good_dm_performance'] = True
    rew = subj_df.reward.sum()
    if rew<=40: # person was worse than chance
        full_data_df.loc[full_data_df.participant==pid, 'good_dm_performance']=False
    
for pid in learn_df.participant.unique():
    subj_df = learn_df[learn_df.participant==pid]
    
    # Add a filter for people with bad performance
    learn_df.loc[learn_df.participant==pid, 'good_dm_performance'] = True
    rew = subj_df.reward.sum()
    if rew<=40: # person was worse than chance
        learn_df.loc[learn_df.participant==pid, 'good_dm_performance']=False

In [ ]:
full_data_df[full_data_df.participant==pid].reward.sum()

In [ ]:
full_data_df.participant.unique().shape

In [ ]:
full_data_df[full_data_df.good_dm_performance].participant.unique().shape

In [ ]:
full_data_df[full_data_df.good_dm_performance==False].participant.unique().shape

Out of N=109, N=87 (~80%) did the learning task well. 

In [ ]:
full_data_df[full_data_df['condition']=='day_2'].participant.unique().shape

Out of N=109, 97 (~89%) did the delayed memory task.

In [ ]:
full_data_df[(full_data_df.good_dm_performance) & (full_data_df['condition']=='day_2')].participant.unique().shape

All of the subjects who did the learning task well also returned for the delayed memory task. 

In [ ]:
full_data_df = full_data_df[full_data_df.good_dm_performance]

In [ ]:
full_data_df['gamble_rt_log'] = np.log(full_data_df.gamble_rt)
full_data_df['recog_rt_log'] = np.log(full_data_df.recog_rt)


In [ ]:
sns.regplot(data=full_data_df, x='auc', y='crit')


## Parameter recovery

This is a parallelized version of the code. 

For maximum effect, I have to REQUEST lots of nodes when I start this instance!!! 

In [ ]:
rw_model = RWModel()
wsls_model = WSLSModel()
bayes_model = BayesModel()
rwfictive1_model = RWModel_Fictive1()
# rwfictive2_model = RWModel_Fictive2()
rwsigned_model = RWModel_Signed()
ewa_model = EWA()

model_types = {'RWModel': rw_model,
         'WSLS': wsls_model, 
         'Bayes': bayes_model, 
         'RWModel_Fictive1': rwfictive1_model,
         'RWModel_Signed': rwsigned_model,
         'EWAModel': ewa_model}

# sim_alpha = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_beta = np.abs(np.random.normal(loc=0, scale=7, size=n_subj))
# sim_bias = np.random.beta(a=1.1, b=1.1, size=n_subj)
# sim_alpha_fictive = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_alpha_pos = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_alpha_neg = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_alpha_rpe = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_alpha_pm = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_phi = np.random.beta(a=1.2, b=1.2, size=n_subj)
# sim_beta = np.abs(np.random.normal(loc=0, scale=7, size=n_subj))
# sim_rho = np.random.beta(a=1.2, b=1.2, size=n_subj)



In [ ]:
import dask.dataframe as dd
from dask.distributed import wait, Client, LocalCluster

In [ ]:
LocalCluster?

In [ ]:
cluster = LocalCluster(name='parameter_recovery', n_workers=12, memory_limit='auto')
client = Client(cluster)
sims = 200

In [ ]:
from typing import Iterable
from dask.distributed import Future
def filter_futures(futures: Iterable[Future], status: Iterable[str] = ["finished"], return_mask=False):
    """
    Helper function to filter futures objects. By default returns only
    futures with status "finished". These can be gathered from distributed
    memory without error. If return_mask, returns (filtered_futures, mask) where mask is a list of booleans 
    signifying which of the futures had the status in question.
    """
    filtered = [f for f in futures if f.status in status]
    if return_mask:
        mask = [f.status in status for f in futures]
        return filtered, mask
    else: return filtered


In [ ]:
[x[0] for x in getmembers(param_recov_utils) if x[0].endswith('parallel')]

In [ ]:
np.random.seed(0)
n_rep = 200
sim_p_switch = np.random.uniform(0, 1, n_rep)

In [ ]:
all_dfs.pid.unique().shape

In [ ]:
# param_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/RW_Model_Fictive1_recov_parallel.csv')


In [ ]:
# run parameter recovery (Just for Bayes data)
all_param_recov_names = [x[0] for x in getmembers(param_recov_utils) if x[0].endswith('parallel')][3:4]
all_param_recov_funs = [x[1] for x in getmembers(param_recov_utils) if x[0].endswith('parallel')][3:4]
n_rep = 200
for ix, fun in enumerate(all_param_recov_funs):
#     futures = client.map(fun, range(sims))
#     wait(futures)
#     results = dd.from_delayed(filter_futures(futures))
#     param_recov_df = results.compute()
    param_recov_df = fun(np.arange(n_rep))
    param_recov_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{all_param_recov_names[ix]}.csv', index=False)
# client.close()
# client.shutdown() 

In [ ]:
[x[0] for x in getmembers(param_recov_utils) if x[0].endswith('parallel')][3:4]

In [ ]:
print('done')

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_parameter_recovery.pdf') as pdf:

    for model_name in [x[0] for x in getmembers(param_recov_utils) if x[0].endswith('parallel')]: 
        try:
            param_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{model_name}.csv')
        except:
            continue
        num_params = param_df.columns.shape[0]//2

        # Plot 
        f, ax = plt.subplots(1, 3, dpi=300, figsize=(12,4))

        for ix, param in enumerate(param_df.columns[:num_params].tolist()): 
            if num_params > 1:
                ax[ix].set_title(f'{model_name}')
                ax[ix].plot(param_df[f'sim_{param}'], param_df[f'{param}'], 'o', color='darkred', alpha=.7)
                if 'beta' in param: 
                    ax[ix].plot([0, 20], [0, 20], 'k--')
                else: 
                    ax[ix].plot([-0.2, 1.2], [-0.2, 1.2], 'k--')
                ax[ix].set_xlabel(f'true {param}')
                ax[ix].set_ylabel(f'fitted {param}')
                f.tight_layout()
            else: 
                ax[ix].set_title(f'{model_name}')
                ax[ix].plot(param_df[f'sim_{param}'], param_df[f'{param}'], 'o', color='darkred', alpha=.7)
                if 'beta' in param: 
                    ax[ix].plot([0, 20], [0, 20], 'k--')
                else: 
                    ax[ix].plot([-0.2, 1.2], [-0.2, 1.2], 'k--')
                ax[ix].set_xlabel(f'true {param}')
                ax[ix].set_ylabel(f'fitted {param}')
                f.tight_layout
        pdf.savefig()
        


In [ ]:
param_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/RWModel_Fictive1_param_recov_parallel.csv')


pearsonr(param_df.sim_beta,
        param_df.beta)



## Compute all models

In [ ]:
import os

In [ ]:
rw_model = RWModel()
wsls_model = WSLSModel()
rwfictive1_model = RWModel_Fictive1()
rwunsigned_model = RWModel_Unsigned()
bayes_model = BayesModel()
ewa_model = EWA()
vkfmodel = VKFModel()

model_types = {'Bayes': bayes_model,
               'RWModel': rw_model,
               'WSLS': wsls_model,
               'VKF': vkfmodel, 
               'RWF1': rwfictive1_model,
               'RWUS': rwunsigned_model,
               'EWA': ewa_model}

load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/DelayedMemoryTest'
num_sub = len(learn_df[learn_df['good_dm_performance']].participant.unique())


for ix in range(num_sub): 
    update_progress(ix/num_sub)
    model_res = {f'{x}':[] for x in model_types.keys()}
    model_divergences = {f'{x}':[] for x in model_types.keys()}
    model_params = {f'{x}':[] for x in model_types.keys()}

    pid = learn_df[learn_df['good_dm_performance']].participant.unique()[ix]
#     if os.path.isfile(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv'):
#         print('already ran this participant')
#         continue
    
#     # Catch the extras 
#     if pid not in full_data_df.groupby('participant').sum()[full_data_df.groupby('participant').sum().reward==40].reset_index().participant.values:
#         continue
        
    actions = learn_df[learn_df.participant==pid].gamble.values
    rewards = learn_df[learn_df.participant==pid].reward.values   
    
    # Make the model 

    for model_key, model in model_types.items(): 
        m = model.make_model(actions, rewards)
        #     Fit the data 
        idata, callback = model.fit(m, tune=1000, target_accept=0.88, cores=None)
        if callback.count > 0:
            idata, callback = model.fit(m, tune=1000, target_accept=0.98, cores=None)
        
        model_res[model_key] = idata
        model_dict = model.compute_posterior_mean_param(idata)
        model_params[model_key] = model_dict
        model_divergences[model_key] = callback.count

    df_comp_loo = az.compare(model_res, ic='waic')
#     add subject name 
    df_comp_loo['pid'] = pid
    df_comp_loo['divergences'] = np.nan
    for key in model_divergences.keys():
        df_comp_loo['divergences'][df_comp_loo.index==key] = model_divergences[key]
#     df_comp_loo['elpd_diff'][df_comp_loo.divergences>0] = df_comp_loo['elpd_diff'].max() + 1
    df_comp_loo.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
    pd.DataFrame.from_dict(model_params).transpose().to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_fit_params_unsigned.csv')
    
    


In [ ]:
print('done')

The above took about 60 hours to complete

In [ ]:
# Model Frequency: 
# aggregate the data
dfs = []
for pid in learn_df[learn_df['good_dm_performance']].participant.unique():
    df_comp_loo = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
    # re-rank the diverging models as tied for worst 
#     df_comp_loo['rank'][df_comp_loo.divergences>0] = np.nan
#         # re-rank non-diverging models ranked 4th as 3rd 
#         df_comp_loo['rank'][(df_comp_loo.divergences==0) & (df_comp_loo['rank']==4)] = 3
    dfs.append(df_comp_loo)
df_comp_loos = pd.concat(dfs)
df_comp_loos.rename(columns={'Unnamed: 0':'model'}, inplace=True)

# Process the divergent models by setting them to the worst rank and moving everything else down a rank
df_comp_loo['rank'][df_comp_loo.divergences>0] = np.nan
# df_comp_loos['rank'][df_comp_loos.model.isin(['RWF1', 'RWS', 'EWA'])] = np.nan

grouped_df = df_comp_loos.sort_values('rank').groupby('pid').apply(pd.DataFrame.head, n=1).drop(columns=['pid']).reset_index()


In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(5,3))
sns.countplot(data=grouped_df, x='model')

In [ ]:
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/plots'
# with PdfPages(f'{save_dir}/EMU_model_comparison.pdf') as pdf:
    
# aggregate the data
dfs = []
for pid in learn_df[learn_df['good_dm_performance']].participant.unique():
    df_comp_loo = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
    # re-rank the diverging models as tied for worst 
#         df_comp_loo['rank'][df_comp_loo.divergences>0] = 4
#         # re-rank non-diverging models ranked 4th as 3rd 
#         df_comp_loo['rank'][(df_comp_loo.divergences==0) & (df_comp_loo['rank']==4)] = 3
    dfs.append(df_comp_loo)
df_comp_loos = pd.concat(dfs)
df_comp_loos.rename(columns={'Unnamed: 0':'model'}, inplace=True)

df_comp_loos['elpd_waic_raw']= -2*df_comp_loos['elpd_waic'] # comptue the actual information crtierion

f, ax = plt.subplots(1,1, dpi=300, figsize=(8,4))
# sns.barplot(data=df_comp_loos[df_comp_loos.divergences==0], x='elpd_waic', y='model', orient='h', errorbar='se',
#            order = ['Bayes', 'RWF1',  'EWA', 'RWS','RWModel', 'VKF', 'WSLS'], estimator='sum')
sns.boxplot(data=df_comp_loos, y='elpd_waic_raw', x='model',  order = ['RWF1', 'Bayes',  'VKF', 'WSLS', 'RWS', 'RWModel', 'EWA'], showcaps=False,boxprops={'facecolor':'None'}, showfliers=False,whiskerprops={'linewidth':0})
sns.stripplot(data=df_comp_loos, y='elpd_waic_raw', x='model', order = ['RWF1', 'Bayes',  'VKF', 'WSLS',  'RWS', 'RWModel', 'EWA'], color='k', size=2, alpha=0.5)
# Get the handles and labels. For this example it'll be 2 tuples
# of length 4 each.
handles, labels = ax.get_legend_handles_labels()
plt.xlabel('model')
plt.ylabel("WAIC")
#     pdf.savefig()
    

In [ ]:
# aggregate the fit data
all_dfs = []
all_files = glob(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/*_model_fit_params.csv')
for file_name in all_files:
    pid = file_name.split('/')[-1].split('_')[0]
    df_comp_param = pd.read_csv(file_name)
    df_comp_param.rename(columns={'Unnamed: 0':'model'}, inplace=True)
    df_comp_param['pid'] = pid
    all_dfs.append(df_comp_param)
# df_comp_params = pd.concat(dfs)

In [ ]:
param_df = pd.concat(all_dfs)

In [ ]:
RWF_df = param_df[param_df.model=='RWF1'][['alpha', 'beta']]
RWF_df['alpha'] = RWF_df['alpha'].apply(lambda x: float(x.strip('][')[6:].split(', ')[0]))
RWF_df['beta'] = RWF_df['beta'].apply(lambda x: float(x.strip('][')[6:].split(', ')[0]))


In [ ]:
Bayes_df = param_df[['p_switch', 'p_reward', 'beta']].dropna()

In [ ]:
Bayes_df['p_switch'] = Bayes_df['p_switch'].apply(lambda x: float(x.strip('][')[6:].split(', ')[0]))
Bayes_df['p_reward'] = Bayes_df['p_reward'].apply(lambda x: float(x.strip('][')[6:].split(', ')[0]))
Bayes_df['beta'] = Bayes_df['beta'].apply(lambda x: float(x.strip('][')[6:].split(', ')[0]))


In [ ]:
sns.histplot(data=Bayes_df, x='beta')

In [ ]:
Bayes_df

In [ ]:
params = {'p_switch': Bayes_df.iloc[0]['p_switch'],
          'p_reward': Bayes_df.iloc[0]['p_reward'],
          'beta': Bayes_df.iloc[0]['beta'],
          'p_noisy': 1e-5}

In [ ]:
bayes_model = BayesModel()
a, r = bayes_model.simulate(n_trials=80, mu=[0.8, 0.2], params=params)

In [ ]:
full_data_df[]

In [ ]:
plt.plot(learn_df[learn_df.participant=='6290c65ed46e5dce4f16bb92'].gamble, 'o-')

In [ ]:
plt.plot(a, 'o-')

In [ ]:
ax = sns.jointplot(data= Bayes_df, x='p_switch', 
                   y='p_reward', 
                   height=8, ratio=2, kind='kde', shade=True, cmap='Spectral', 
                   fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
# ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
# ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)

In [ ]:
ax = sns.jointplot(data= RWF_df, x='alpha', 
                   y='beta', 
                   height=8, ratio=2, kind='kde', shade=True, cmap='Spectral', 
                   fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)

In [ ]:
# rw_model = RWModel()
# wsls_model = WSLSModel()
# bayes_model = BayesModel()
# vkf_model = EWA()

# model_types = {'RWModel_Signed': rwsigned_model,
#                'Bayes': bayes_model,
#                'RWModel': rw_model,
#          'WSLS': wsls_model}

# load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/DelayedMemoryTest'
# # dm_df_all = pd.read_csv(f'{load_dir}/full_data.csv')
# # num_sub = len(dm_df_all[dm_df_all['good_dm_performance']].participant.unique())

# num_sub = len(full_data_df[full_data_df['good_dm_performance']].participant.unique())

# for ix in range(num_sub): 
#     update_progress(ix/num_sub)
#     model_res = {f'{x}':[] for x in model_types.keys()}
#     model_divergences = {f'{x}':[] for x in model_types.keys()}
#     model_params = {f'{x}':[] for x in model_types.keys()}

#     pid = full_data_df[full_data_df['good_dm_performance']].participant.unique()[ix]
    
#     # Catch the extras 
#     if pid not in full_data_df.groupby('participant').sum()[full_data_df.groupby('participant').sum().reward==40].reset_index().participant.values:
#         continue
    

#     for model_key, model in model_types.items(): 
#         mod = model.make_model(full_data_df[full_data_df.participant==pid].gamble.dropna().values, 
#                           full_data_df[full_data_df.participant==pid].reward.dropna().values)
#         idata, callback = model.fit(mod, tune=1500, target_accept=0.9, cores=None)
#         model_res[model_key] = idata
#         model_dict = model.compute_posterior_mean_param(idata)
#         model_params[model_key] = model_dict
#         model_divergences[model_key] = callback.count

#     df_comp_loo = az.compare(model_res, ic='loo')
# #     add subject name 
#     df_comp_loo['pid'] = pid
#     df_comp_loo['divergences'] = np.nan
#     for key in model_divergences.keys():
#         df_comp_loo['divergences'][df_comp_loo.index==key] = model_divergences[key]
#     df_comp_loo['elpd_diff'][df_comp_loo.divergences>0] = df_comp_loo['elpd_diff'].max() + 1
#     df_comp_loo.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
#     pd.DataFrame.from_dict(model_params).transpose().to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_fit_params.csv')
    
    


In [ ]:
pid

In [ ]:
ix

In [ ]:
df_comp = df_comp_loo.reset_index(names='model').sort_values(by='elpd_diff')

In [ ]:
if (df_comp[df_comp.model=='RWModel'].elpd_diff <= df_comp.iloc[0].se).values:
    print(True)

In [ ]:
(df_comp[df_comp.model=='RWModel'].elpd_diff <= df_comp.iloc[0].se).values[0]

I would like to do this in Dask to parallelize across different subjects. However, the dask parallelization does not play well with the parallelization already in play in pymc sampling across multiple chains across multiple cores. If I set cores=1 I could probably run it, but for that would need a lot more cores overall to mitigate the slow-down in sampling. 

UPDATE: 

Running in dask causes lots of problems with the pymc sampling or something. Avoid for now. 


In [ ]:
# def run_models(ix):
    
#     import numpy as np
#     import matplotlib.pyplot as plt
#     import seaborn as sns
#     import scipy
#     import glob as glob
#     import pymc as pm
#     from pytensor import tensor as pt 
#     import pytensor
#     import arviz as az
#     import pandas as pd
#     import sys 
#     import statsmodels.formula.api as smf
#     import statsmodels.api as sm
#     from scipy.stats import chi2, ttest_ind, pearsonr
#     from scipy.io import loadmat
#     from matplotlib.backends.backend_pdf import PdfPages


#     sys.path.append('/hpc/users/qasims01/resources/ImgRatingSorter/')
#     from util import dprime
#     sys.path.append('/hpc/users/qasims01/resources/utils/RLModels_K99/scripts')
# #     from model_utils import *
#     from RWModel import RWModel
#     from WSLSModel import WSLSModel
#     from BayesModel import BayesModel
#     from RWModel_Fictive1 import RWModel_Fictive1
#     from RWModel_Fictive2 import RWModel_Fictive2
#     from RWModel_Signed import RWModel_Signed
#     from EWAModel import EWA
    
#     base_dir = '/sc/arion'

#     rw_model = RWModel()
#     wsls_model = WSLSModel()
#     bayes_model = BayesModel()
#     rwfictive1_model = RWModel_Fictive1()
#     rwfictive2_model = RWModel_Fictive2()
#     rwsigned_model = RWModel_Signed()
#     ewa_model = EWA()

#     model_types = {'RWModel': rw_model,
#              'WSLS': wsls_model, 
#              'Bayes': bayes_model, 
#              'RWModel_Fictive1': rwfictive1_model,
#              'RWModel_Fictive2': rwfictive2_model,
#              'RWModel_Signed': rwsigned_model,
#              'EWAModel': ewa_model}
    
#     model_res = {f'{x}':[] for x in model_types.keys()}
    
#     model_params = {f'{x}':[] for x in model_types.keys()}

    
#     load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/DelayedMemoryTest'

#     dm_df_all = pd.read_csv(f'{load_dir}/full_data.csv')
    
#     pid = dm_df_all[dm_df_all['good_dm_performance']].participant.unique()[ix]

#     for model_key, model in model_types.items(): 
#         mod = model.make_model(dm_df_all[dm_df_all.participant==pid].gamble.dropna().values, 
#                           dm_df_all[dm_df_all.participant==pid].reward.dropna().values)
#         idata = model.fit(mod, tune=1500, target_accept=0.9)
#         model_res[model_key] = idata
#         model_dict = model.compute_posterior_mean_param(idata)
#         model_params[model_key] = model_dict
        
#     df_comp_loo = az.compare(model_res, ic='waic')
#     # add subject name 
#     df_comp_loo['pid'] = pid
#     df_comp_loo.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
#     joblib.dump(model_res, f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_fit_params')
    
# #     return df_comp_loo

In [ ]:
# import dask.dataframe as dd
# from dask.distributed import wait, Client, LocalCluster

In [ ]:
# import dask.dataframe as dd
# from dask.distributed import wait, Client, LocalCluster
# cluster = LocalCluster()
# client = Client(cluster)

In [ ]:
# client = Client(local_directory='/sc/arion/scratch/qasims01/dask_space/')

In [ ]:
# num_sub = len(full_data_df[full_data_df['good_dm_performance']].participant.unique())

# futures = client.map(run_models, range(num_sub))
# wait(futures)
# # results = dd.from_delayed(futures)

In [ ]:
full_data_df.participant.unique().shape

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_model_comparison.pdf') as pdf:
    
    # aggregate the data
    dfs = []
    for pid in full_data_df[full_data_df['good_dm_performance']].participant.unique():
        df_comp_loo = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
        dfs.append(df_comp_loo)
    df_comp_loos = pd.concat(dfs)
    df_comp_loos.rename(columns={'Unnamed: 0':'model'}, inplace=True)

    f, ax = plt.subplots(1,1, dpi=300, figsize=(8,4))
    sns.barplot(data=df_comp_loos, x='elpd_diff', y='model', orient='h', ci=68, 
                order = ['RWModel', 'RWModel_Signed', 'Bayes', 'WSLS'])
#     sns.stripplot(data=df_comp_loos[df_comp_loos.warning==False], x='elpd_diff', y='model', orient='h', order = ['RWModel', 'Bayes', 'WSLS',
#       'RWModel_Fictive1', 'RWModel_Signed'], color='k', size=2, alpha=0.75)
    # Get the handles and labels. For this example it'll be 2 tuples
    # of length 4 each.
    handles, labels = ax.get_legend_handles_labels()
    sns.despine()
    plt.ylabel('model')
    plt.xlabel(u'\u0394 elpd')
    pdf.savefig()
    
    

In [ ]:
from scipy.stats import ttest_rel
ttest_rel(df_comp_loos[df_comp_loos.model=='RWModel'].elpd_diff,
          df_comp_loos[df_comp_loos.model=='RWModel_Signed'].elpd_diff)


In [ ]:
best_fit_model_key = df_comp_loo.reset_index(names='model').iloc[0].model

In [ ]:
best_fit_model_key

In [ ]:
model_names_in_order = ['RWModel', 'WSLS', 'Bayes', 'RWModel_Fictive1', 'RWModel_Signed']


In [ ]:
model_names_in_order.index(best_fit_model_key)

In [ ]:
model_names_in_order.index

## Confusion matrix

In [ ]:
all_files = glob(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/*_model_fit_params.csv')
confusion_mat = model_recov_utils.model_recov(np.arange(len(all_files)), T=80, mu = np.array([0.8, 0.2]))


In [ ]:
print(confusion_mat)

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_model_recovery.pdf') as pdf:
    
#     confusion_mat = np.load(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/trunc_confusion_mat.npy')
    confusion_mat = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/trunc_confusion_mat')
    model_names_in_order = ['RWF1', 'WSLS', 'Bayes', 'VKF']
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(9, 9))
    sns.heatmap(data=confusion_mat.drop(columns=['Unnamed: 0']), cmap='Spectral_r', annot=True, cbar=False)
    plt.gca().invert_yaxis()
    ax.set(xticklabels=model_names_in_order, yticklabels=model_names_in_order)
    pdf.savefig()
    

In [ ]:
confusion_mat

Troubleshoot the model recovery process: 

In [ ]:
all_files = glob(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/*_model_fit_params.csv')

x = np.arange(len(all_files))

# set base directory 
base_dir = '/sc/arion'

# aggregate the fit data
all_dfs = []
all_files = glob(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/*_model_fit_params.csv')
for file_name in all_files:
    pid = file_name.split('/')[-1].split('_')[0]
    df_comp_param = pd.read_csv(file_name)
    df_comp_param.rename(columns={'Unnamed: 0':'model'}, inplace=True)
    df_comp_param['pid'] = pid
    all_dfs.append(df_comp_param)
# df_comp_params = pd.concat(dfs)

# set task simulation parameters 
T = 80
mu = np.array([0.8, 0.2])

# Check if we are running as an input into Dask for parallel processing,
# or just letting it go sequentially 

if hasattr(x, "__len__"): 
    # Input could be an array of values 
    if len(x) > 1: 
        run_type = 'sequential'
    else: 
        run_type = 'parallel'
else: 
    # Input is a scalar
    run_type = 'parallel'

# initialize models 
# model_names_in_order = ['RWModel', 'WSLS', 'Bayes', 'RWModel_Fictive1', 'RWModel_Signed', 'EWAModel']
model_names_in_order = ['RWModel', 'WSLS', 'Bayes', 'RWModel_Fictive1', 'RWModel_Signed']

# rw_model = RWModel()
wsls_model = WSLSModel()
# bayes_model = BayesModel()
# rwfictive1_model = RWModel_Fictive1()
rwsigned_model = RWModel_Signed()
# ewa_model = EWA()

# model_types = {'RWModel': rw_model,
#         'WSLS': wsls_model, 
#         'Bayes': bayes_model, 
#         'RWModel_Fictive1': rwfictive1_model,
#         'RWModel_Signed': rwsigned_model,
#         'EWAModel': ewa_model}

rw_model = RWModel()
bayes_model = BayesModel()
rwfictive1_model = RWModel_Fictive1()

model_types = {'RWModel': rw_model,
        'WSLS': wsls_model, 
        'RWModel_Fictive1': rwfictive1_model,
        'Bayes': bayes_model, 
        'RWModel_Signed': rwsigned_model}

model_res = {f'{x}':[] for x in model_types.keys()}

confusion_mat = np.zeros([len(model_names_in_order), len(model_names_in_order)])

In [ ]:
ix = 0
param_df = all_dfs[ix]
pid = all_files[ix].split('/')[-1].split('_')[0]

# if os.path.isfile(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/trunc_confusion_mat_{ix}.npy'):
#     confusion_mat = np.load(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/trunc_confusion_mat_{ix}.npy')
#     continue

actions_dict = {f'{x}': np.nan for x in model_types.keys()}
rewards_dict = {f'{x}': np.nan for x in model_types.keys()}
for mod in model_names_in_order: 
    # Simulate actions using each model using real params
    # For some reason the parameters got encoded as strings lol oops

    if mod == 'RWModel':
        params = {'alpha': float(param_df[param_df.model==mod].alpha.iloc[0].strip('][')[6:].split(', ')[0]),
        'beta': float(param_df[param_df.model==mod].beta.iloc[0].strip('][')[6:].split(', ')[0])}
    elif mod == 'WSLS':
        params = {'bias': float(param_df[param_df.model==mod].bias.iloc[0].strip('][')[6:].split(', ')[0])}
    elif mod == 'Bayes':
        params = {'p_switch': float(param_df[param_df.model==mod].p_switch.iloc[0].strip('][')[6:].split(', ')[0]), 
        'p_reward': float(param_df[param_df.model==mod].p_reward.iloc[0].strip('][')[6:].split(', ')[0]), 
        'p_noisy': 1e-5, 
        'beta': float(param_df[param_df.model==mod].beta.iloc[0].strip('][')[6:].split(', ')[0])}
    elif mod == 'RWModel_Fictive1':
        params = {'alpha': float(param_df[param_df.model==mod].alpha.iloc[0].strip('][')[6:].split(', ')[0]),
        'beta': float(param_df[param_df.model==mod].beta.iloc[0].strip('][')[6:].split(', ')[0])}
    elif mod == 'RWModel_Signed':
         params = {'alpha_pos': float(param_df[param_df.model==mod].alpha_pos.iloc[0].strip('][')[6:].split(', ')[0]),
        'alpha_neg': float(param_df[param_df.model==mod].alpha_neg.iloc[0].strip('][')[6:].split(', ')[0]),
        'beta': float(param_df[param_df.model==mod].beta.iloc[0].strip('][')[6:].split(', ')[0])} 
    elif mod == 'EWAModel':
        params = {'phi': float(param_df[param_df.model==mod].phi.iloc[0].strip('][')[6:].split(', ')[0]),
        'beta': float(param_df[param_df.model==mod].beta.iloc[0].strip('][')[6:].split(', ')[0]),
        'rho': float(param_df[param_df.model==mod].rho.iloc[0].strip('][')[6:].split(', ')[0])}

    a,r = model_types[mod].simulate(n_trials=T, mu=mu, params=params)

    actions_dict[mod] = a
    rewards_dict[mod] = r
    
    

In [ ]:
        # BIC_scores = [] 
        for mod_key_real in model_names_in_order[0:1]:
            model_res = {f'{x}':[] for x in model_names_in_order}
            for mod_key_fitted in model_names_in_order:
                fitted_model = model_types[mod_key_fitted].make_model(actions_dict[mod_key_real], rewards_dict[mod_key_real])
                idata = model_types[mod_key_fitted].fit(fitted_model, tune=1500, target_accept=0.85, cores=None)
                model_res[mod_key_fitted] = idata
                # BIC_scores.append(len(x) * np.nanmean(model_res[mod_key_fitted].log_likelihood.like.values) + len(model_types[mod_key_fitted].model_params) * np.log(len(x)))

            df_comp = az.compare(model_res, ic='waic', method="BB-pseudo-BMA")
            # Get rid of models with warnings 
            df_comp = df_comp[df_comp.warning==False]
            # If there isn't a big difference between the real model and the best model, choose the real model
            if (df_comp.loc[f'{mod_key_real}'].elpd_diff<1) & (df_comp.reset_index(names='model').iloc[0].se>2):
                best_fit_model_key=mod_key_real
            else:
                best_fit_model_key = df_comp.reset_index(names='model').iloc[0].model
            # best_fit_ix = model_names_in_order.index(best_fit_model_key)
            # best_fit_model_key = model_names_in_order[np.argmin(BIC_scores)]
        #     best_fit_ix = model_names_in_order.index(best_fit_model_key)
            confusion_mat.loc[mod_key_real, best_fit_model_key] = confusion_mat.loc[mod_key_real, best_fit_model_key] + 1

In [ ]:
df_comp

## Add model parameters to dataframes

In [ ]:
# aggregate the fit data
all_dfs = []
all_files = glob(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/*_model_fit_params.csv')
for file_name in all_files:
    pid = file_name.split('/')[-1].split('_')[0]
    df_comp_param = pd.read_csv(file_name)
    df_comp_param.rename(columns={'Unnamed: 0':'model'}, inplace=True)
    df_comp_param['pid'] = pid
    all_dfs.append(df_comp_param)
# df_comp_params = pd.concat(dfs)

In [ ]:
# Now, model the data with the best model and save params
model = RWModel_Fictive1()

full_data_df['alpha'] = np.nan
full_data_df['beta'] = np.nan
# full_data_df['prpe']= np.nan
# full_data_df['alpha_neg'] = np.nan
# full_data_df['nrpe']= np.nan
# full_data_df['rpe']= np.nan

learn_df['alpha'] = np.nan
learn_df['beta'] = np.nan
# learn_df['prpe']= np.nan
# learn_df['alpha_neg'] = np.nan
# learn_df['nrpe']= np.nan
learn_df['rpe']= np.nan

# Loop through subjects: 
num_sub = len(full_data_df[full_data_df['good_dm_performance']].participant.unique())
for ix in range(num_sub):
    pid = full_data_df[full_data_df['good_dm_performance']].participant.unique()[ix]
    param_df = all_dfs[ix]
    params = {'alpha': float(param_df[param_df.model=='RWF1'].alpha.iloc[0].strip('][')[6:].split(', ')[0]),
              'beta': float(param_df[param_df.model=='RWF1'].beta.iloc[0].strip('][')[6:].split(', ')[0])}

    update_progress(ix / num_sub)
    
    actions = learn_df[learn_df.participant==pid].gamble.values
    rewards = learn_df[learn_df.participant==pid].reward.values   
    
    # Make the model 
#     m = model.make_model(actions, rewards)
    
    # Fit the data 
#     idata = model.fit(model=m, tune=1500, target_accept=0.9, cores=None)
    
    # Get the mean of the posterior
#     model_dict = model.compute_posterior_mean_param(idata)

    rpe = model.simulate(n_trials=80, actions=actions, rewards=rewards, params=params) 

    full_data_df['alpha'][full_data_df.participant==pid] = params['alpha']
    full_data_df['beta'][full_data_df.participant==pid] = params['beta']
#     full_data_df['rpe'][full_data_df.participant==pid] = rpe
    
    learn_df['alpha'][learn_df.participant==pid] = params['alpha']
    learn_df['beta'][learn_df.participant==pid] = params['beta']
#     learn_df['prpe'][learn_df.participant==pid] = prpe
#     learn_df['nrpe'][learn_df.participant==pid] = nrpe
    learn_df['rpe'][learn_df.participant==pid] = rpe
 
full_data_df = full_data_df.merge(learn_df[['participant', 'img_path', 'rpe']], on=['participant', 'img_path'], how='left')
full_data_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_df_RWF1.csv')
learn_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_RWF1.csv')


## --> Enter here!

In [ ]:
full_data_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_df_RWF1.csv')
learn_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_RWF1.csv')


In [ ]:
full_data_df.keys()

In [ ]:
# code in the log reaction time for gamble choces 
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

# code in the prior trial's RPE 
# learn_df['prior_rpe'] = np.roll(learn_df.rpe, 1)
# learn_df['prior_rpe'].iloc[0] = np.nan

In [ ]:
# Load all the data and merge it (improve this):
# Because we merged data earlier to compute memory metrics, drop the columns so we can re-merge now and get RPE 

drop_cols = [x for x in np.intersect1d(learn_df.columns, full_data_df.columns) if x not in ['img_path', 'participant']]

full_data_df.drop(columns=drop_cols, inplace=True)

combined_df = full_data_df.merge(learn_df, on=['img_path', 'participant'], how='left', suffixes=['_mem', '_gamble'])

# Filter out people by performance in both tasks 
high_perf_subs = combined_df.groupby('participant').agg({'good_dm_performance': np.nanmean}).reset_index()
high_perf_subs = high_perf_subs[high_perf_subs.good_dm_performance==1].participant.values
high_perf_mask = combined_df.participant.apply(lambda x: True if x in high_perf_subs else False)
plot_df = combined_df[high_perf_mask]
# plot_df = plot_df[plot_df.dprime>0]

# Let's log-transform the reaction times 
plot_df['conf_rt_log'] = np.log(plot_df.conf_rt)
plot_df['recog_rt_log'] = np.log(plot_df.recog_rt)

# Load the psychometric data 
psych_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/'
psych_data = pd.read_csv(f'{psych_dir}/MemoryBandit_REDCap_Delay.csv')
psych_data.rename(columns={'prolific_pid': 'participant'}, inplace=True)
psych_data = psych_data[(np.isnan(psych_data['imc_1']) & np.isnan(psych_data['imc_2']))]
# drop all uninvolved pts 
psych_data = psych_data[psych_data['participant'].isin(plot_df.participant.unique())].reset_index(drop=True)
# add the condition and remove the first session for people who did this 3x 
psych_data['condition'] = [f'day_{x }' for x in psych_data.groupby(['participant']).cumcount().values+1]
repeat_pts = psych_data[psych_data.groupby(['participant']).cumcount()==2].participant.unique()
psych_data[psych_data.participant.isin(repeat_pts) & (psych_data.condition=='day_1')] = np.nan
psych_data.condition[psych_data.participant.isin(repeat_pts) & (psych_data.condition=='day_2')] = 'day_1'
psych_data.condition[psych_data.participant.isin(repeat_pts) & (psych_data.condition=='day_3')] = 'day_2'

# 
# Load demographic information: 
demo_file = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/demographics.csv'
demo_data = pd.read_csv(demo_file, index_col=None, header=0)
demo_data.rename(columns={'Participant id': 'participant'}, inplace=True)

# Merge the combined df with the psych and demo data 
full_data = plot_df.merge(demo_data.drop_duplicates(subset=['participant']), on=['participant'])

full_data['gamble_rt_log'] = np.log(full_data.gamble_rt)
full_data['recog_rt_log'] = np.log(full_data.recog_rt)


# # Get rid of slow-ass trials
# full_data = full_data[full_data.gamble_rt<10]
# full_data = full_data[full_data.recog_rt<20]
# # Get rid of too-fast trials 
# # full_data = full_data[full_data.gamble_rt>=0.2]
# full_data = full_data[full_data.recog_rt>=0.4]

# note that this can throw away high performance trials.... 


full_data_psych = full_data.merge(psych_data, on=['participant', 'condition'])
full_data_psych['depr_state'] = np.nan
full_data_psych['anx_s_state'] = np.nan
full_data_psych['anx_t_state'] = np.nan
full_data_psych['ocd_state'] = np.nan


full_data_psych['depr_state'][full_data_psych.condition=='day_1'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_1']['sds_score'], q=3, labels=['low', 'med', 'high'])
full_data_psych['anx_s_state'][full_data_psych.condition=='day_1'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_1']['stai_s_score'], q=3, labels=['low', 'med', 'high'])
full_data_psych['anx_t_state'][full_data_psych.condition=='day_1'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_1']['stai_t_score'], q=3, labels=['low', 'med', 'high'])
full_data_psych['ocd_state'][full_data_psych.condition=='day_1'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_1']['ocirscore'],q=3, labels=['low', 'med', 'high'])


full_data_psych['depr_state'][full_data_psych.condition=='day_2'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_2']['sds_score'], q=3, labels=['low', 'med', 'high'])
full_data_psych['anx_s_state'][full_data_psych.condition=='day_2'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_2']['stai_s_score'], q=3, labels=['low', 'med','high'])
full_data_psych['anx_t_state'][full_data_psych.condition=='day_2'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_2']['stai_t_score'], q=3, labels=['low', 'med','high'])
full_data_psych['ocd_state'][full_data_psych.condition=='day_2'] = pd.qcut(full_data_psych[full_data_psych.condition=='day_2']['ocirscore'],q=3, labels=['low', 'med', 'high'])



In [ ]:
full_data[full_data.condition=='day_1'].Age

In [ ]:
subj_level_df_mean = learn_df.groupby('participant').mean().reset_index()
subj_level_df_mean['num_trials'] = learn_df.groupby('participant').count().reset_index().learn_trials
subj_level_df_mean['points'] = subj_level_df_mean.apply(lambda x: x.reward * 100 * 80, axis=1)

In [ ]:
subj_level_df_mean = plot_df.groupby(['participant', 'condition']).mean().reset_index()
subj_level_df_mean['points'] = subj_level_df_mean.apply(lambda x: x.reward * 100 * 80, axis=1)
    
    

# Analysis

In [ ]:
# What is the distribution of reaction times for gambling and memory? 

with PdfPages(f'{save_dir_plots}/Expw_RTHists.pdf') as pdf:    
    f, ax = plt.subplots(1,3, dpi=300, figsize=(9,2))
    sns.histplot(data=full_data[full_data.condition=='day_1'], x='gamble_rt_log', ax=ax[0], kde=True, color=sequential_colors[0], element='step', label='gambling')
    sns.histplot(data=full_data[full_data.condition=='day_1'], x='recog_rt_log', ax=ax[1], kde=True, color=sequential_colors[0], element='step', label='memory')
    sns.regplot(data=full_data[full_data.condition=='day_1'].groupby('participant').mean(), x='gamble_rt_log', y='recog_rt_log', order=1, ax=ax[2], 
                scatter_kws={'s':6, 'alpha':0.5}, color=sequential_colors[0])
    sns.histplot(data=full_data[full_data.condition=='day_2'], x='recog_rt_log', ax=ax[1], kde=True, color=sequential_colors[1], element='step', label='memory')
    sns.regplot(data=full_data[full_data.condition=='day_2'].groupby('participant').mean(), x='gamble_rt_log', y='recog_rt_log', order=1, ax=ax[2], 
                scatter_kws={'s':6, 'alpha':0.5}, color=sequential_colors[1])
#     ax[0].set_xlim([0, 10])
#     ax[1].set_xlim([0, 10])
    ax[0].set_xlabel('log gamble RT')
    ax[1].set_xlabel('log memory RT')            
    ax[2].set_xlabel('log gamble RT')
    ax[2].set_ylabel('log memory RT')
    sns.despine()
#     plt.legend()
    pdf.savefig()
    
    

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_RewardHists.pdf') as pdf:    
    f, ax = plt.subplots(1,3, dpi=300, figsize=(9,2))
    sns.histplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='points', ax=ax[0], kde=True, color=sequential_colors[0], element='step', label='gambling')
    sns.histplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='dprime', ax=ax[1], kde=True, color=sequential_colors[0], element='step', label='memory')
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='points', y='dprime', order=1, ax=ax[2], 
                scatter_kws={'s':6, 'alpha':0.5}, color=sequential_colors[0])
    sns.histplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_2'], x='dprime', ax=ax[1], kde=True, color=sequential_colors[1], element='step', label='memory')
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_2'], x='points', y='dprime', order=1, ax=ax[2], 
                scatter_kws={'s':6, 'alpha':0.5}, color=sequential_colors[1])
    ax[2].set_xlabel('reward (a.u.)')
    ax[2].set_ylabel('dprime')
    sns.despine()
    pdf.savefig()

In [ ]:
print(pearsonr(subj_level_df_mean[subj_level_df_mean.condition=='day_1'].dropna(subset='dprime').points,
        subj_level_df_mean[subj_level_df_mean.condition=='day_1'].dropna(subset='dprime').dprime))

In [ ]:
print(pearsonr(subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dropna(subset='dprime').points,
        subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dropna(subset='dprime').dprime))

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_dprime_between_days.pdf') as pdf:
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
    sns.pointplot(data=subj_level_df_mean, x='condition', y='dprime', hue='participant', color ='k', ax=ax)
    plt.legend([],[], frameon=False)
    plt.setp(ax.collections, alpha=0) #for the markers
    plt.setp(ax.lines, alpha=.1)       #for the lines
    ax.plot(np.zeros(len(subj_level_df_mean[subj_level_df_mean.condition=='day_1'].dprime)), 
           subj_level_df_mean[subj_level_df_mean.condition=='day_1'].dprime, 'o', color=sequential_colors[0])

    ax.plot(np.ones(len(subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dprime)), 
           subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dprime, 'o', color=sequential_colors[1])

    sns.despine()
    plt.hlines(0, plt.xlim()[0],  plt.xlim()[1], color='red')
    ax.set(xticklabels=['Day 1', 'Day 2'], ylabel="memory (d')")
    pdf.savefig()

In [ ]:
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/plots'
# with PdfPages(f'{save_dir}/model_parameters_delay.pdf') as pdf:
#     ax = sns.pairplot(data=learn_df[learn_df.good_dm_performance].groupby('participant').mean()[['alpha_pos', 'alpha_neg', 'beta']], 
#                  kind='kde', height=4, aspect=1.5,
#                       plot_kws={'cmap':'Spectral_r', 
#                                        'fill':True, 
#                                        'cbar':True})
#     ax.axes[0,1].set(xticks=np.linspace(0,1, 5))
#     # ax.axes[0,2].set(xticks=np.linspace(0,1, 5))

#     ax.axes[1,0].set(xticks=np.linspace(0,1, 5))


#     ax.axes[2,0].set(xticks=np.linspace(0,1, 5))
#     ax.axes[2,1].set(xticks=np.linspace(0,1, 5))
    
#     pdf.savefig()



In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Fig_RWParams.pdf') as pdf: 
    ax = sns.jointplot(data = learn_df[learn_df.good_dm_performance].groupby('participant').mean(), 
                       x='alpha', y='beta', height=8, ratio=2, kind='kde', shade=True, cmap='Spectral_r', 
                       fill=True, cbar=True)

    plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
    # get the current positions of the joint ax and the ax for the marginal x
    pos_joint_ax = ax.ax_joint.get_position()
    pos_marg_x_ax = ax.ax_marg_x.get_position()
    # reposition the joint ax so it has the same width as the marginal x ax
    ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
    # reposition the colorbar using new x positions and y positions of the joint ax
    ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
    ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
    ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)
    pdf.savefig()



## Day 1

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_learn_behav.pdf') as pdf:
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,2))
    sns.lineplot(data=learn_df[learn_df['good_dm_performance']], x='learn_trials', y= 'male', color='orange', ax=ax, errorbar='se')
    sns.lineplot(data=learn_df[learn_df['good_dm_performance']], x='learn_trials', y= 'female', color='purple', ax=ax, errorbar='se')
    plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
    plt.ylabel('p(draw)')
    plt.title(f'subject performance')
    sns.despine()
    pdf.savefig()
    
    

I think it's likely that after 3 reversals, people tend to re-evaluate their priors on number of reversals or probability of reversals, OR maybe they're just too bored/tired to be paying attention to reversals??? 



The latter seems most likely to me. 

Another possibility - they get VERY good in block 3 - maybe they are loathe to leave that strategy?

In [ ]:
# How do the parameter estimates correspond with performance? 

with PdfPages(f'{save_dir_plots}/Exp2_Fig_RewardBetaCorr.pdf') as pdf:    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], 
                x='beta', y='points', color = 'black', ci=95, scatter_kws={'s':6,'alpha':0.5})
    sns.despine()
    ax.set_ylim([4000, 8000])
    ax.set_ylabel('subject reward')
    pdf.savefig()


In [ ]:
pearsonr(subj_level_df_mean[subj_level_df_mean.condition=='day_1'].beta, 
         subj_level_df_mean[subj_level_df_mean.condition=='day_1'].points)

In [ ]:
# How do the parameter estimates correspond with performance? 

with PdfPages(f'{save_dir_plots}/Exp2_Fig_RewardAlphaCorr.pdf') as pdf:    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], 
                x='alpha', y='points', color = 'black', ci=95, scatter_kws={'s':6,'alpha':0.5})
    sns.despine()
    ax.set_ylim([4000, 8000])
    ax.set_ylabel('subject reward')
    pdf.savefig()


In [ ]:
pearsonr(subj_level_df_mean[subj_level_df_mean.condition=='day_1'].alpha, 
         subj_level_df_mean[subj_level_df_mean.condition=='day_1'].points)

## Memory

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Fig_DprimeBetaCorr.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='beta', 
                y='dprime', color = sequential_colors[0], ci=95, scatter_kws={'s':6, 'alpha':0.5})
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_2'], x='beta', 
                y='dprime', color = sequential_colors[1], ci=95, scatter_kws={'s':6, 'alpha':0.5})
    sns.despine()
    ax.set_ylabel('subject dprime')
    
    pdf.savefig(f)

In [ ]:
pearsonr(subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dropna(subset='dprime').beta, 
         subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dropna(subset='dprime').dprime)

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Fig_DprimeAlphaCorr.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='alpha', 
                y='dprime', color = sequential_colors[0], ci=95, scatter_kws={'s':6, 'alpha':0.5})
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_2'], x='alpha', 
                y='dprime', color = sequential_colors[1], ci=95, scatter_kws={'s':6, 'alpha':0.5})
    sns.despine()
    ax.set_ylabel('subject dprime')
    
    pdf.savefig(f)

In [ ]:
pearsonr(subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dropna(subset='dprime').alpha, 
         subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dropna(subset='dprime').dprime)

In [ ]:
# trial-level rpe-memory correlation

with PdfPages(f'{save_dir_plots}/Exp2_Fig_MemoryRPECorr.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(4,2))
    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_1')], x='rpe', y='hits', color = sequential_colors[0], 
                logistic=True, ci=None, scatter=False, ax=ax[0])
    for pid in plot_df[plot_df._merge=='both'].participant.unique():
        sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_1') & (plot_df.participant==pid)], 
                    x='rpe', y='hits', color = sequential_colors[0], ax=ax[0],
                logistic=True, ci=None, scatter=False, line_kws={'alpha':0.03})
#     ax.set_ylim([0.45, .65])
    ax[0].set_ylabel('p(hit)')
    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_2')], x='rpe', y='hits', color = sequential_colors[1], 
                logistic=True, ci=None, scatter=False, ax=ax[1])
    for pid in plot_df[plot_df._merge=='both'].participant.unique():
        sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_2') & (plot_df.participant==pid)], 
                    x='rpe', y='hits', color = sequential_colors[1], ax=ax[1],
                logistic=True, ci=None, scatter=False, line_kws={'alpha':0.03})
    f.tight_layout()
    sns.despine()
    pdf.savefig()


In [ ]:
# trial-level rpe-memory correlation

with PdfPages(f'{save_dir_plots}/Exp2_Fig_MemoryImageDprimeCorr.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(4,2))
    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_1')], x='image_dprime', y='hits', color = sequential_colors[0], 
                logistic=True, ci=None, scatter=False, ax=ax[0])
    for pid in plot_df[plot_df._merge=='both'].participant.unique():
        sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_1') & (plot_df.participant==pid)], 
                    x='image_dprime', y='hits', color = sequential_colors[0], ax=ax[0],
                logistic=True, ci=None, scatter=False, line_kws={'alpha':0.03})
#     ax.set_ylim([0.45, .65])
    ax[0].set_ylabel('p(hit)')
    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_2')], x='image_dprime', y='hits', color = sequential_colors[1], 
                logistic=True, ci=None, scatter=False, ax=ax[1])
    for pid in plot_df[plot_df._merge=='both'].participant.unique():
        sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_2') & (plot_df.participant==pid)], 
                    x='image_dprime', y='hits', color = sequential_colors[1], ax=ax[1],
                logistic=True, ci=None, scatter=False, line_kws={'alpha':0.03})
        
    f.tight_layout()
    sns.despine()
    pdf.savefig()


In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_HRandFAR_between_days.pdf') as pdf:
    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
    sns.regplot(data = subj_level_df_mean[subj_level_df_mean.condition=='day_1'], 
                x= 'hit_rate', y='false_alarm_rate', color=sequential_colors[0])
    sns.regplot(data = subj_level_df_mean[subj_level_df_mean.condition=='day_2'], 
                    x= 'hit_rate', y='false_alarm_rate', color=sequential_colors[1])
    sns.despine()
    pdf.savefig()

# Do some modeling of the effect of features across days

In [ ]:
# Prep a list to store results 
all_res = [] 

full_data['img_cat'] = full_data.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

model_df = full_data[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                           'alpha', 'beta', 'Age', 'Sex', 'reward', 'participant', 'hit_rate', 'condition',
                         'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm', 'alpha', 'beta']

# Set response variable as int
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'{factor}'] = model_df[f'{factor}'].fillna(0)
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
#     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

      
model_df = model_df[model_df.condition=='day_1']

# model_df = model_df[model_df.depr_state!='med']
      
# cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)

In [ ]:
model_df.participant.unique().shape

In [ ]:
y = 'memory_outcome'
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
within_subject_X = ['zrpe', 'zpm']
Intx = ['zalpha:zrpe', 'zbeta:zrpe']
rand_effect = ['participant']
categorical = None

label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day1'

# Drop nan data
model_df = model_df.dropna(subset=X)

model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
                           priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
                           target_accept=0.95, model_fam='bernoulli', 
                           output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                           return_model=True,
                           categorical_baseline=True, 
                           save_model_res = True, 
                           label=label)

f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
sns.despine()
plot_res(results, X, ax=ax)



In [ ]:
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day1'
results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_model')

with PdfPages(f'{save_dir_plots}/Exp2_Posteriors_Day1.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
    ax.set(xlabel='Weight', ylabel='Parameter')
    sns.despine()
    plot_res(results, X, ax=ax)
    
    pdf.savefig(f)

In [ ]:
# Prep a list to store results 
all_res = [] 

full_data['img_cat'] = full_data.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

model_df = full_data[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                           'alpha', 'beta', 'Age', 'Sex', 'reward', 'participant', 'hit_rate', 'condition',
                         'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm', 'alpha', 'beta']

# Set response variable as int
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'{factor}'] = model_df[f'{factor}'].fillna(0)
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
    # Set nan's to 0 to capture the LACK of RPE 
#     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

      
model_df = model_df[model_df.condition=='day_2']

# model_df = model_df[model_df.depr_state!='med']
      
# cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)

In [ ]:
y = 'memory_outcome'
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
within_subject_X = ['zrpe', 'zpm']
Intx = ['zalpha:zrpe', 'zbeta:zrpe']
rand_effect = ['participant']
categorical = None

label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day2'

# Drop nan data
model_df = model_df.dropna(subset=X)

model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
                           priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
                           target_accept=0.95, model_fam='bernoulli', 
                           output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                           return_model=True,
                           categorical_baseline=True, 
                           save_model_res = True, 
                           label=label)

f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
sns.despine()
plot_res(results, X, ax=ax)



In [ ]:
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day2'
results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_model')

with PdfPages(f'{save_dir_plots}/Exp2_Posteriors_Day2.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
    ax.set(xlabel='Weight', ylabel='Parameter')
    sns.despine()
    plot_res(results, X, ax=ax)
    
    pdf.savefig(f)

In [ ]:
# Prep a list to store results 
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day1'
result_csv = pd.read_csv(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_summary.csv')
result_csv.rename(columns={'Unnamed: 0': 'parameter'}, inplace=True)
pattern = r'\[(\d.*?)]'

# use the re.search() function to search for the first occurrence of the pattern
# match = re.search(pattern, input_string)
result_csv['participant'] = result_csv.parameter.str.extract(pattern)

pm_index = result_csv[result_csv.parameter.str.contains('zpm')].dropna(subset=['participant'])['parameter'].index 
result_csv.loc[pm_index, 'parameter'] = 'zpm_weight'
rpe_index = result_csv[result_csv.parameter.str.contains('zrpe')].dropna(subset=['participant'])['parameter'].index 
result_csv.loc[rpe_index, 'parameter'] = 'zrpe_weight'


all_res = [] 


model_df = full_data[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                      'alpha', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'condition',
                      'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge']].copy()

# 'alpha_pos', 'alpha_neg',

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm']

# Set response variable as in 
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

independent_vars = ['zrpe', 'zpm']

model_df = model_df[model_df.condition=='day_1']

all_participants = model_df.participant.unique()
model_betas = [] 

for pid in all_participants:
    dfs = [] 
    models = {} 
    subj_df = model_df[model_df.participant==pid]
    
    # make regression weights based on response var frequency
    y_ind = subj_df.memory_outcome.astype(int)
    recip_freq = 1. / np.bincount(y_ind)
    recip_freq /= np.mean(recip_freq)
    wts = recip_freq[y_ind]
    
    if (y_ind.sum()<10)  | (y_ind.sum()>110):
        print(subj_df.dprime.mean(), y_ind.sum())
        print('model failure - not enough successes')
        continue
    else:
        # Make the model formula
        formula = f'{dependent_var} ~ 1+'+'+'.join(independent_vars)
        # Make the model
        model = smf.glm(formula = formula, 
                        family=sm.families.Binomial(), 
                        data=subj_df, 
                        freq_weights=wts)
        # Set the baseline 
        model._ynames_map = {0: 0, 1: 1}
        # Fit the model 
        try:
            results = model.fit()
        except:
            print('model failure - perfect separation')
            continue

    # Collect the individual subject model results. 
    glm_df_temp = results.params.to_dict()
    glm_df = pd.DataFrame(columns=['subj_dprime', 'participant', 'alpha_pos', 'alpha_neg', 'beta'])
    glm_df['GLMBeta'] = list(glm_df_temp.values())
    glm_df['GLMFactor'] = list(glm_df_temp.keys())
    glm_df['subj_dprime'] = subj_df.dprime.mean()
    glm_df['subj_reward'] = full_data[full_data.participant==pid].reward.sum()
    glm_df['subj_rpe'] = subj_df.rpe.mean() 
    glm_df['subj_HR'] = subj_df.hit_rate.mean()
    glm_df['subj_FAR'] = subj_df.false_alarm_rate.mean()
    glm_df['subj_gamble_rt'] = subj_df.gamble_rt_log.mean()
    glm_df['subj_recog_rt'] = subj_df.recog_rt_log.mean()
#     glm_df['alpha_pos'] = subj_df.alpha_pos.mean()
#     glm_df['alpha_neg'] = subj_df.alpha_neg.mean()
    glm_df['alpha'] = subj_df.alpha.mean()
    glm_df['beta'] = subj_df.beta.mean()
    glm_df['participant'] = pid
    glm_df['zrpe_weight'] = glm_df_temp['zrpe']
    glm_df['zpm_weight'] = glm_df_temp['zpm']
    glm_df['zrpe_weight_mm'] = result_csv[(result_csv['participant'].str.contains(pid)) & (result_csv.parameter=='zrpe_weight')]['mean'].values[0]
    glm_df['zpm_weight_mm'] = result_csv[(result_csv['participant'].str.contains(pid)) & (result_csv.parameter=='zpm_weight')]['mean'].values[0]


    model_betas.append(glm_df)
day1_df = pd.concat(model_betas)
day1_df.reset_index(drop=True, inplace=True)

# # Group the data by participant 
day1_df_subj = day1_df.groupby(['participant']).mean().reset_index()
day1_df_subj = day1_df_subj.merge(day1_df['participant'], on='participant').drop_duplicates()





In [ ]:
# sns.regplot(data=subj_mean_df, x='subj_dprime', y='aucs')
with PdfPages(f'{save_dir_plots}/Exp2_Model1_Reward_Weights_Day1.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=False)
    sns.regplot(data=day1_df_subj, x='subj_reward', y='zrpe_weight_mm', ci=None, ax=ax[0], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[0])
    sns.regplot(data=day1_df_subj, x='subj_reward', y='zpm_weight_mm', ci=None, ax=ax[1], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[0])
    ax[0].set(xlabel = 'reward (points)', ylabel='RPE contribution to memory')
    ax[1].set(xlabel = 'reward (points)', ylabel='PM contribution to memory')
    sns.despine()
    f.tight_layout()
    pdf.savefig()

In [ ]:
# sns.regplot(data=subj_mean_df, x='subj_dprime', y='aucs')
with PdfPages(f'{save_dir_plots}/Exp2_Fig_LogRegress_WeightsvMemory_Day1.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=True)
    sns.regplot(data=day1_df_subj, x='zrpe_weight_mm', y='subj_dprime', ci=None, ax=ax[0], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[0])
    sns.regplot(data=day1_df_subj, x='zpm_weight_mm', y='subj_dprime', ci=None, ax=ax[1], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[0])
    ax[0].set(xlabel = 'RPE contribution to memory', ylabel='memory (d)')
    ax[1].set(xlabel = 'PM contribution to memory', ylabel='memory (d)')
    sns.despine()
    f.tight_layout()
    pdf.savefig()

In [ ]:
# Prep a list to store results 
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day2'
result_csv = pd.read_csv(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_summary.csv')
result_csv.rename(columns={'Unnamed: 0': 'parameter'}, inplace=True)
pattern = r'\[(\d.*?)]'

# use the re.search() function to search for the first occurrence of the pattern
# match = re.search(pattern, input_string)
result_csv['participant'] = result_csv.parameter.str.extract(pattern)

pm_index = result_csv[result_csv.parameter.str.contains('zpm')].dropna(subset=['participant'])['parameter'].index 
result_csv.loc[pm_index, 'parameter'] = 'zpm_weight'
rpe_index = result_csv[result_csv.parameter.str.contains('zrpe')].dropna(subset=['participant'])['parameter'].index 
result_csv.loc[rpe_index, 'parameter'] = 'zrpe_weight'


all_res = [] 


model_df = full_data[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                      'alpha', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'condition',
                      'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge']].copy()

# 'alpha_pos', 'alpha_neg',

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm']

# Set response variable as in 
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

independent_vars = ['zrpe', 'zpm']

model_df = model_df[model_df.condition=='day_2']

all_participants = model_df.participant.unique()
model_betas = [] 

for pid in all_participants:
    dfs = [] 
    models = {} 
    subj_df = model_df[model_df.participant==pid]
    
    # make regression weights based on response var frequency
    y_ind = subj_df.memory_outcome.astype(int)
    recip_freq = 1. / np.bincount(y_ind)
    recip_freq /= np.mean(recip_freq)
    wts = recip_freq[y_ind]
    
    if (y_ind.sum()<10)  | (y_ind.sum()>110):
        print(subj_df.dprime.mean(), y_ind.sum())
        print('model failure - not enough successes')
        continue
    else:
        # Make the model formula
        formula = f'{dependent_var} ~ 1+'+'+'.join(independent_vars)
        # Make the model
        model = smf.glm(formula = formula, 
                        family=sm.families.Binomial(), 
                        data=subj_df, 
                        freq_weights=wts)
        # Set the baseline 
        model._ynames_map = {0: 0, 1: 1}
        # Fit the model 
        try:
            results = model.fit()
        except:
            print('model failure - perfect separation')
            continue

    # Collect the individual subject model results. 
    glm_df_temp = results.params.to_dict()
    glm_df = pd.DataFrame(columns=['subj_dprime', 'participant', 'alpha_pos', 'alpha_neg', 'beta'])
    glm_df['GLMBeta'] = list(glm_df_temp.values())
    glm_df['GLMFactor'] = list(glm_df_temp.keys())
    glm_df['subj_dprime'] = subj_df.dprime.mean()
    glm_df['subj_reward'] = full_data[full_data.participant==pid].reward.sum()
    glm_df['subj_rpe'] = subj_df.rpe.mean() 
    glm_df['subj_HR'] = subj_df.hit_rate.mean()
    glm_df['subj_FAR'] = subj_df.false_alarm_rate.mean()
    glm_df['subj_gamble_rt'] = subj_df.gamble_rt_log.mean()
    glm_df['subj_recog_rt'] = subj_df.recog_rt_log.mean()
#     glm_df['alpha_pos'] = subj_df.alpha_pos.mean()
#     glm_df['alpha_neg'] = subj_df.alpha_neg.mean()
    glm_df['alpha'] = subj_df.alpha.mean()
    glm_df['beta'] = subj_df.beta.mean()
    glm_df['participant'] = pid
    glm_df['zrpe_weight'] = glm_df_temp['zrpe']
    glm_df['zpm_weight'] = glm_df_temp['zpm']
    glm_df['zrpe_weight_mm'] = result_csv[(result_csv['participant'].str.contains(pid)) & (result_csv.parameter=='zrpe_weight')]['mean'].values[0]
    glm_df['zpm_weight_mm'] = result_csv[(result_csv['participant'].str.contains(pid)) & (result_csv.parameter=='zpm_weight')]['mean'].values[0]


    model_betas.append(glm_df)
day2_df = pd.concat(model_betas)
day2_df.reset_index(drop=True, inplace=True)

# # Group the data by participant 
day2_df_subj = day2_df.groupby(['participant']).mean().reset_index()
day2_df_subj = day2_df_subj.merge(day2_df['participant'], on='participant').drop_duplicates()





In [ ]:
# sns.regplot(data=subj_mean_df, x='subj_dprime', y='aucs')
with PdfPages(f'{save_dir_plots}/Exp2_Model1_Reward_Weights_Day2.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=False)
    sns.regplot(data=day2_df_subj, x='subj_reward', y='zrpe_weight_mm', ci=None, ax=ax[0], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[1])
    sns.regplot(data=day2_df_subj, x='subj_reward', y='zpm_weight_mm', ci=None, ax=ax[1], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[1])
    ax[0].set(xlabel = 'reward (points)', ylabel='RPE contribution to memory')
    ax[1].set(xlabel = 'reward (points)', ylabel='PM contribution to memory')
    sns.despine()
    f.tight_layout()
    pdf.savefig()

In [ ]:
# sns.regplot(data=subj_mean_df, x='subj_dprime', y='aucs')
with PdfPages(f'{save_dir_plots}/Exp2_Fig_LogRegress_WeightsvMemory_Day2.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=True)
    sns.regplot(data=day2_df_subj, x='zrpe_weight_mm', y='subj_dprime', ci=None, ax=ax[0], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[1])
    sns.regplot(data=day2_df_subj, x='zpm_weight_mm', y='subj_dprime', ci=None, ax=ax[1], 
                scatter_kws={'alpha':0.33}, color=sequential_colors[1])
    ax[0].set(xlabel = 'RPE contribution to memory', ylabel='memory (d)')
    ax[1].set(xlabel = 'PM contribution to memory', ylabel='memory (d)')
    sns.despine()
    f.tight_layout()
    pdf.savefig()

In [ ]:
combined_df = day1_df_subj.merge(day2_df_subj, on=['participant'], suffixes=['_day1', '_day2'])

In [ ]:
combined_df.keys()

In [ ]:
# sns.regplot(data=subj_mean_df, x='subj_dprime', y='aucs')
with PdfPages(f'{save_dir_plots}/Exp2_Day1vDay2_ModelWeights.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=False)
    sns.regplot(data=combined_df, x='zrpe_weight_mm_day1', y='zrpe_weight_mm_day2', ci=None, ax=ax[0], 
                scatter_kws={'alpha':0.33}, color='black')
    sns.regplot(data=combined_df, x='zpm_weight_mm_day1', y='zpm_weight_mm_day2', ci=None, ax=ax[1], 
                scatter_kws={'alpha':0.33}, color='black')
    ax[0].set(xlabel = 'RPE contribution to memory', ylabel='memory (d)')
    ax[1].set(xlabel = 'PM contribution to memory', ylabel='memory (d)')
    sns.despine()
    f.tight_layout()
    pdf.savefig()

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Day1vDay2_Memory.pdf') as pdf:    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3), sharey=False)

    sns.regplot(data=combined_df, x='subj_dprime_day1', y='subj_dprime_day2', ci=None, 
                    scatter_kws={'alpha':0.33}, color='black', ax=ax)
    
    ax.plot([0, 3], [0, 3], 'k--')

    sns.despine()
    f.tight_layout()
    pdf.savefig()

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_memory_behav.pdf') as pdf:
    
    day1_df = full_data_df[full_data_df.condition=='day_1']
    signal = day1_df[(day1_df.hits==1) | ((day1_df.misses==1)) ].recog_choice_bool
    noise = day1_df[(day1_df.correct_rejections==1) | ((day1_df.false_alarms==1)) ].recog_choice_bool
    n_samples = len(signal)
    responses = np.concatenate((signal, noise))  # Combine signal and noise responses
    true_labels = np.concatenate((np.ones(n_samples), np.zeros(n_samples)))  # 1 for signal, 0 for noise

    # Compute ROC curve and AUC score
    fpr, tpr, _ = roc_curve(true_labels, responses)
    roc_auc = auc(fpr, tpr)

    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
    # Plot the ROC curve
    ax.plot(fpr, tpr, color=sequential_colors[0], lw=2, label='ROC curve (area = %0.2f)' % roc_auc)

    day2_df = full_data_df[full_data_df.condition=='day_2']
    signal = day2_df[(day2_df.hits==1) | ((day2_df.misses==1)) ].recog_choice_bool
    noise = day2_df[(day2_df.correct_rejections==1) | ((day2_df.false_alarms==1)) ].recog_choice_bool
    n_samples = len(signal)
    responses = np.concatenate((signal, noise))  # Combine signal and noise responses
    true_labels = np.concatenate((np.ones(n_samples), np.zeros(n_samples)))  # 1 for signal, 0 for noise

    # Compute ROC curve and AUC score
    fpr, tpr, _ = roc_curve(true_labels, responses)
    roc_auc = auc(fpr, tpr)
    # Plot the ROC curve
    ax.plot(fpr, tpr, color=sequential_colors[1], lw=2, label='ROC curve (area = %0.2f)' % roc_auc)




    ax.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Receiver Operating Characteristic (ROC) Curve')
    ax.legend(loc="lower right")
    
    pdf.savefig()

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_memory_behav_conf.pdf') as pdf:
    f, ax = plt.subplots(1,4, dpi=300, figsize=(6,2))

    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_1')], x='conf_choice', y='hits', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.2}, y_jitter=.05, x_jitter=.05, ax=ax[0], color=sequential_colors[0])
    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_2')], x='conf_choice', y='hits', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.2}, y_jitter=.05, x_jitter=.05, ax=ax[0], color=sequential_colors[1])
    ax[0].set_ylabel('p(hit)')
    ax[0].set_xlabel('confidence')

    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_1')], x='conf_choice', y='misses', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.1}, y_jitter=.05, x_jitter=.05, ax=ax[1], color=sequential_colors[0])
    sns.regplot(data=plot_df[(plot_df._merge=='both') & (plot_df.condition=='day_2')], x='conf_choice', y='misses', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.1}, y_jitter=.05, x_jitter=.05, ax=ax[1], color=sequential_colors[1])
    ax[1].set_ylabel('p(miss)')
    ax[1].set_xlabel('confidence')

    sns.regplot(data=plot_df[(plot_df._merge=='left_only') & (plot_df.condition=='day_1')], x='conf_choice', y='false_alarms', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.1}, y_jitter=.05, x_jitter=.05, ax=ax[2], color=sequential_colors[0])
    sns.regplot(data=plot_df[(plot_df._merge=='left_only') & (plot_df.condition=='day_2')], x='conf_choice', y='false_alarms', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.1}, y_jitter=.05, x_jitter=.05, ax=ax[2], color=sequential_colors[1])
    ax[2].set_ylabel('p(false alarm)')
    ax[2].set_xlabel('confidence')

    sns.regplot(data=plot_df[(plot_df._merge=='left_only') & (plot_df.condition=='day_1')], x='conf_choice', y='correct_rejections', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.1}, y_jitter=.05, x_jitter=.05, ax=ax[3], color=sequential_colors[0])
    sns.regplot(data=plot_df[(plot_df._merge=='left_only') & (plot_df.condition=='day_2')], x='conf_choice', y='correct_rejections', logistic=True, ci=None, n_boot=None, scatter_kws={'s':1, 'alpha':0.1}, y_jitter=.05, x_jitter=.05, ax=ax[3], color=sequential_colors[1])
    ax[3].set_ylabel('p(correct rejection)')
    ax[3].set_xlabel('confidence')

    f.tight_layout()
    pdf.savefig()


Can we model confidence? 

UPDATE: YES - it doesn't really change the results


In [ ]:
# # Prep a list to store results 
# all_res = [] 

# full_data['img_cat'] = full_data.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

# model_df = full_data[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
#                            'alpha', 'beta', 'Age', 'Sex', 'reward', 'participant', 'hit_rate', 'condition', 'conf_choice',
#                          'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

# model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# # Make a new response variable, memory outcome 
# model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
#                                        'false_alarms']].idxmax(1).to_frame('memory_outcome')

# print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

      
# # ADD CONFIDENCE TO MEMORY CODING
      
# # CODE THE VARIABLES: 
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = -1
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
# model_df['memory_outcome'] = model_df['memory_outcome'] * model_df['conf_choice']
      

# # Subject level traits need to be filled across NaNs
# model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
# model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
# # model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# # model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
# model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
# model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# # Define the dependent variable and the independent variables
# dependent_var = 'memory_outcome'
# independent_vars = ['rpe', 'pm', 'alpha', 'beta']

# # Set response variable as int
# model_df[dependent_var] = model_df[dependent_var].astype(int)
# # Zscore independent variables: 
# for factor in independent_vars: 
#     # Set nan's to 0 to capture the LACK of RPE 
#     model_df[f'{factor}'] = model_df[f'{factor}'].fillna(0)
#     model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
# #     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

      
# model_df = model_df[model_df.condition=='day_1']

# # model_df = model_df[model_df.depr_state!='med']
      
# # cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# # model_df.depr_state = model_df.depr_state.astype(cat_type)

In [ ]:
# y = 'memory_outcome'
# X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
# within_subject_X = ['zrpe', 'zpm']
# Intx = ['zalpha:zrpe', 'zbeta:zrpe']
# rand_effect = ['participant']
# categorical = None

# label = (f"{y}_confidence" + "_{}"*len(X)).format(*X) + 'exp2_day1'

# # Drop nan data
# model_df = model_df.dropna(subset=X)

# model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
#                            priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
#                            target_accept=0.95, model_fam='gaussian', 
#                            output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
#                            return_model=True,
#                            categorical_baseline=True, 
#                            save_model_res = True, 
#                            label=label)

# f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
# ax.set(xlabel='Weight', ylabel='Parameter')
# sns.despine()
# plot_res(results, X, ax=ax)



In [ ]:
# # Prep a list to store results 
# all_res = [] 

# full_data['img_cat'] = full_data.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

# model_df = full_data[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
#                            'alpha', 'beta', 'Age', 'Sex', 'reward', 'participant', 'hit_rate', 'condition', 'conf_choice',
#                          'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

# model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# # Make a new response variable, memory outcome 
# model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
#                                        'false_alarms']].idxmax(1).to_frame('memory_outcome')

# print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

      
# # ADD CONFIDENCE TO MEMORY CODING
      
# # CODE THE VARIABLES: 
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = -1
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
# model_df['memory_outcome'] = model_df['memory_outcome'] * model_df['conf_choice']
      

# # Subject level traits need to be filled across NaNs
# model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
# model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
# # model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# # model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
# model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
# model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# # Define the dependent variable and the independent variables
# dependent_var = 'memory_outcome'
# independent_vars = ['rpe', 'pm', 'alpha', 'beta']

# # Set response variable as int
# model_df[dependent_var] = model_df[dependent_var].astype(int)
# # Zscore independent variables: 
# for factor in independent_vars: 
#     # Set nan's to 0 to capture the LACK of RPE 
#     model_df[f'{factor}'] = model_df[f'{factor}'].fillna(0)
#     model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
# #     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

      
# model_df = model_df[model_df.condition=='day_2']

# # model_df = model_df[model_df.depr_state!='med']
      
# # cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# # model_df.depr_state = model_df.depr_state.astype(cat_type)

In [ ]:
# y = 'memory_outcome'
# X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
# within_subject_X = ['zrpe', 'zpm']
# Intx = ['zalpha:zrpe', 'zbeta:zrpe']
# rand_effect = ['participant']
# categorical = None

# label = (f"{y}_confidence" + "_{}"*len(X)).format(*X) + 'exp2_day2'

# # Drop nan data
# model_df = model_df.dropna(subset=X)

# model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
#                            priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
#                            target_accept=0.95, model_fam='gaussian', 
#                            output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
#                            return_model=True,
#                            categorical_baseline=True, 
#                            save_model_res = True, 
#                            label=label)

# f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
# ax.set(xlabel='Weight', ylabel='Parameter')
# sns.despine()
# plot_res(results, X, ax=ax)



In [ ]:
# def plot_2d_regression(x,y,z, plane_color='r'): 
#     fig = plt.figure()
#     ax = fig.add_subplot(111, projection='3d')
#     ax.scatter3D(x, 
#                  y, 
#                  z, alpha = 1, color='k');


#     n_samples = len(x)
#     # Create the design matrix
#     X = np.vstack((x, y)).T
#     X = np.hstack((np.ones((n_samples, 1)), X))

#     from sklearn.linear_model import HuberRegressor
# #     TheilSenRegressor

#     # Fit a linear regression model
#     X = np.column_stack((x, y))
#     reg = HuberRegressor().fit(X, z)
# #     TheilSenRegressor

#     # Get the coefficients of the plane
#     a, b = reg.coef_
#     c = reg.intercept_

#     # Create a meshgrid to plot the plane
#     x_plane, y_plane = np.meshgrid(np.linspace(0, 1, 50), np.linspace(0, 15, 50))
#     z_plane = a * x_plane + b * y_plane + c
    
#     # Calculate the residuals
#     residuals = []
#     z_pred = []
#     for i in range(len(x)):
#         z_predicted = a * x[i] + b * y[i] + c
#         residual = z[i] - z_predicted
#         z_pred.append(z_predicted)
#         residuals.append(residual)
#     r2 = r2_score(z, z_pred)

#     # Plot the plane
#     ax.plot_surface(x_plane, y_plane, z_plane, alpha=0.2, color=plane_color)
#     ax.plot_wireframe(X=x_plane, Y=y_plane, Z=z_plane, rstride=5, cstride=3, linewidth=1, alpha = 1, color=plane_color)   # 3d plot

#     points = [(a,b,c) for a,b,c in zip(x,y,z)]
#     for ix, point in enumerate(points):
#         xs = [point[0], point[0]]
#         ys = [point[1], point[1]]
#         zs = [point[2], point[2] - residuals[ix]]
#         ax.plot(xs, ys, zs, color='k')


#     ax.view_init(10, -60)

#     # Show the plot
#     plt.show()
    
#     return a,b,r2


## Depression

In [ ]:
sub_level_df_psych = full_data_psych.groupby(['participant', 'condition']).mean().reset_index()
sub_level_df_psych['points'] = sub_level_df_psych.reward*100*80

sub_level_df_psych['depr_state'] = np.nan
sub_level_df_psych['anx_s_state'] = np.nan
sub_level_df_psych['anx_t_state'] = np.nan
sub_level_df_psych['ocd_state'] = np.nan

for condition in ['day_1', 'day_2']:

    sub_level_df_psych['depr_state'][sub_level_df_psych.condition==condition] = pd.qcut(sub_level_df_psych['sds_score'][sub_level_df_psych.condition==condition], q=3, labels=['low', 'med', 'high'])
    sub_level_df_psych['anx_s_state'][sub_level_df_psych.condition==condition] = pd.qcut(sub_level_df_psych['stai_s_score'][sub_level_df_psych.condition==condition],  q=3, labels=['low', 'med','high'])
    sub_level_df_psych['anx_t_state'][sub_level_df_psych.condition==condition] = pd.qcut(sub_level_df_psych['stai_t_score'][sub_level_df_psych.condition==condition],  q=3, labels=['low', 'med','high'])
    sub_level_df_psych['ocd_state'][sub_level_df_psych.condition==condition] = pd.qcut(sub_level_df_psych['ocirscore'][sub_level_df_psych.condition==condition],  q=3, labels=['low', 'med','high'])


In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Fig_SDS_hist.pdf') as pdf:    
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,3), sharey=True)
    sns.histplot(data = sub_level_df_psych[(sub_level_df_psych.condition=='day_1')], 
             x='sds_score', 
             kde=True, 
             element='step',
             hue='depr_state',
             ax=ax[0])
    sns.histplot(data = sub_level_df_psych[(sub_level_df_psych.condition=='day_2')], 
             x='sds_score', 
             kde=True, 
             element='step',
             hue='depr_state',
             ax=ax[1])
    sns.despine()
    pdf.savefig()

In [ ]:
f, ax = plt.subplots(1,2, dpi=300, figsize=(8,3))
sns.histplot(data=sub_level_df_psych[(sub_level_df_psych.condition=='day_1')], x='sds_score',  kde=True, element='step', color=sequential_colors[0], ax=ax[0])
sns.histplot(data=sub_level_df_psych[(sub_level_df_psych.condition=='day_2')], x='sds_score', kde=True, element='step',color=sequential_colors[1], ax=ax[1])


Are the scores correlated between days? 

In [ ]:
day1_df = sub_level_df_psych[sub_level_df_psych.condition=='day_1']
day2_df = sub_level_df_psych[sub_level_df_psych.condition=='day_2']

plot_df = day1_df.merge(day2_df, on=['participant'])

In [ ]:
sns.regplot(data=plot_df,
            x='sds_score_x',
           y='sds_score_y', ci=None)

In [ ]:
from scipy.stats import pearsonr

r1, p1 = pearsonr(sub_level_df_psych[sub_level_df_psych.condition=='day_1'].alpha_neg,
          sub_level_df_psych[sub_level_df_psych.condition=='day_1'].sds_score)

In [ ]:
from scipy.stats import pearsonr

r2, p2 = pearsonr(sub_level_df_psych[sub_level_df_psych.condition=='day_1'].alpha_pos,
          sub_level_df_psych[sub_level_df_psych.condition=='day_1'].sds_score)

In [ ]:
from scipy.stats import pearsonr

pearsonr(sub_level_df_psych[sub_level_df_psych.condition=='day_1'].beta,
          sub_level_df_psych[sub_level_df_psych.condition=='day_1'].sds_score)

In [ ]:
compare_corr(r1, r2, 
            sub_level_df_psych[sub_level_df_psych.condition=='day_1'].sds_score.shape[0],
            sub_level_df_psych[sub_level_df_psych.condition=='day_1'].sds_score.shape[0])

In [ ]:
sns.histplot(data=sub_level_df_psych[(sub_level_df_psych.condition=='day_1') & (sub_level_df_psych.depr_state=='low')], x='dprime',  kde=True, element='step',color='gray')
sns.histplot(data=sub_level_df_psych[(sub_level_df_psych.condition=='day_1') & (sub_level_df_psych.depr_state=='high')], x='dprime', kde=True, element='step',color='red')




In [ ]:
sns.histplot(data=sub_level_df_psych[(sub_level_df_psych.condition=='day_2') & (sub_level_df_psych.depr_state=='low')], x='dprime',  kde=True, element='step',color='gray')
sns.histplot(data=sub_level_df_psych[(sub_level_df_psych.condition=='day_2') & (sub_level_df_psych.depr_state=='high')], x='dprime', kde=True, element='step',color='red')




In [ ]:
ttest_ind(sub_level_df_psych[sub_level_df_psych.depr_state=='low'].rpe,
          sub_level_df_psych[sub_level_df_psych.depr_state=='high'].rpe)

In [ ]:
ttest_ind(sub_level_df_psych[sub_level_df_psych.depr_state=='low'].dprime,
          sub_level_df_psych[sub_level_df_psych.depr_state=='high'].dprime)

In [ ]:
    # Decision-making performance: 
    f, ax = plt.subplots(2,2, dpi=300, figsize=(4,4), sharey=True)
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='low'], x='reward',  kde=True, element='step',color='gray', ax=ax[0,0])
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='high'], x='reward', kde=True, element='step',color='red', ax=ax[0,0])

    # Memory performance: 
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='low'], x='dprime',  kde=True, element='step',color='gray', ax=ax[0,1])
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='high'], x='dprime', kde=True, element='step',color='red', ax=ax[0,1])

    # Alpha
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='low'], x='alpha',  kde=True, element='step',color='gray', ax=ax[1,0])
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='high'], x='alpha', kde=True, element='step',color='red', ax=ax[1,0])

    # Beta: 
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='low'], x='beta',  kde=True, element='step',color='gray', ax=ax[1,1])
    sns.histplot(data=sub_level_df_psych[sub_level_df_psych.depr_state=='high'], x='beta', kde=True, element='step',color='red', ax=ax[1,1])

    f.tight_layout()

In [ ]:
sns.regplot(data=sub_level_df_psych, x='sds_score', y='beta')

The way that depressed people use RPEs for memory changes across days

In [ ]:
f, ax = plt.subplots(1,2, dpi=300, figsize=(8,3))
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_1') & (full_data_psych.depr_state=='low')], x='rpe', y='hits', logistic=True, ci=None, n_boot=None, color='gray', ax=ax[0])
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_2') & (full_data_psych.depr_state=='low')], x='rpe', y='hits', logistic=True, ci=None, n_boot=None,  color='gray', ax=ax[1])
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_1') & (full_data_psych.depr_state=='high')], x='rpe', y='hits', logistic=True, ci=None, n_boot=None, color='red',  ax=ax[0])
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_2') & (full_data_psych.depr_state=='high')], x='rpe', y='hits', logistic=True, ci=None, n_boot=None,  color='red',  ax=ax[1])

#     sns.regplot(data=full_data_psych[full_data_psych.anx_t_state=='low'], x='rpe', y='hits', color = 'gray', logistic=True, ci=None, scatter_kws={'s':1, 'alpha':0.2}, y_jitter=.05, x_jitter=.05)
#     sns.regplot(data=full_data_psych[full_data_psych.anx_t_state=='high'], x='rpe', y='hits', color = 'red', logistic=True, ci=None, scatter_kws={'s':1, 'alpha':0.2}, y_jitter=.05, x_jitter=.05)
#     sns.regplot(data=plot_df[plot_df._merge=='both'], x='gamble_x', y='hits', color = 'green', logistic=True, ci=68)

ax[0].set_ylim([0.45, .75])
ax[1].set_ylim([0.45, .75])

# ax.set_ylabel('p(hit)')

In [ ]:
f, ax = plt.subplots(1,2, dpi=300, figsize=(8,3))
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_1') & (full_data_psych.depr_state=='low')], x='image_dprime', y='hits', logistic=True, ci=None, n_boot=None, color='gray', ax=ax[0])
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_2') & (full_data_psych.depr_state=='low')], x='image_dprime', y='hits', logistic=True, ci=None, n_boot=None,  color='gray', ax=ax[1])
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_1') & (full_data_psych.depr_state=='high')], x='image_dprime', y='hits', logistic=True, ci=None, n_boot=None, color='red',  ax=ax[0])
sns.regplot(data=full_data_psych[(full_data_psych.condition=='day_2') & (full_data_psych.depr_state=='high')], x='image_dprime', y='hits', logistic=True, ci=None, n_boot=None,  color='red',  ax=ax[1])

#     sns.regplot(data=full_data_psych[full_data_psych.anx_t_state=='low'], x='rpe', y='hits', color = 'gray', logistic=True, ci=None, scatter_kws={'s':1, 'alpha':0.2}, y_jitter=.05, x_jitter=.05)
#     sns.regplot(data=full_data_psych[full_data_psych.anx_t_state=='high'], x='rpe', y='hits', color = 'red', logistic=True, ci=None, scatter_kws={'s':1, 'alpha':0.2}, y_jitter=.05, x_jitter=.05)
#     sns.regplot(data=plot_df[plot_df._merge=='both'], x='gamble_x', y='hits', color = 'green', logistic=True, ci=68)

ax[0].set_ylim([0.2, .5])
ax[1].set_ylim([0.2, .5])



## Model 1

Depression correlates with less reward, but not memory (d') on either day

In [ ]:
# Prep a list to store results 
all_res = [] 

full_data_psych['img_cat'] = full_data_psych.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

model_df = full_data_psych[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                           'alpha', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'sds_score', 'condition',
                            'depr_state', 'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm', 'alpha', 'beta', 'sds_score']

# Set response variable as int
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'{factor}'] = model_df[f'{factor}'].fillna(0)
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 

#     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

      
model_df = model_df[model_df.condition=='day_1']

# model_df = model_df[model_df.depr_state!='med']
      
# cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)

In [ ]:
y = 'memory_outcome'
X = ['zrpe', 'zpm', 'zsds_score', 'zalpha', 'zbeta']
within_subject_X = ['zrpe', 'zpm']
Intx = ['zsds_score:zrpe', 'zsds_score:zpm']
rand_effect = ['participant']
categorical = None

# Drop nan data
model_df = model_df.dropna(subset=X)

label = (f"{y}" + "_{}"*len(X)).format(*X) + '_experiment2_day1_depr'


model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
                           priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
                           target_accept=0.95, model_fam='bernoulli', 
                           output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                           return_model=True,
                           categorical_baseline=True, 
                           save_model_res = True, 
                           label=label)


f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
sns.despine()
plot_res(results, X, ax=ax)

In [ ]:
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zsds_score', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + '_experiment2_day1_depr'
results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_model')

with PdfPages(f'{save_dir_plots}/Exp2_Posteriors_Day1_Depr.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
    ax.set(xlabel='Weight', ylabel='Parameter')
    sns.despine()
    plot_res(results, X, ax=ax)
    
    pdf.savefig(f)

In [ ]:
# Prep a list to store results 
all_res = [] 

full_data_psych['img_cat'] = full_data_psych.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

model_df = full_data_psych[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                           'alpha', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'sds_score', 'condition',
                            'depr_state', 'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm', 'alpha', 'beta', 'sds_score']

# Set response variable as int
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'{factor}'] = model_df[f'{factor}'].fillna(0)
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
#     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

      
model_df = model_df[model_df.condition=='day_2']

# model_df = model_df[model_df.depr_state!='med']
      
# cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)

In [ ]:
y = 'memory_outcome'
X = ['zrpe', 'zpm', 'zsds_score', 'zalpha', 'zbeta']
within_subject_X = ['zrpe', 'zpm']
Intx = ['zsds_score:zrpe', 'zsds_score:zpm']
rand_effect = ['participant']
categorical = None

# Drop nan data
model_df = model_df.dropna(subset=X)

label = (f"{y}" + "_{}"*len(X)).format(*X) + '_experiment2_day2_depr'


model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
                           priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
                           target_accept=0.95, model_fam='bernoulli', 
                           output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                           return_model=True,
                           categorical_baseline=True, 
                           save_model_res = True, 
                           label=label)


f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
sns.despine()
plot_res(results, X, ax=ax)

In [ ]:
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zsds_score', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + '_experiment2_day2_depr'
results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_model')

with PdfPages(f'{save_dir_plots}/Exp2_Posteriors_Day2_Depr.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
    ax.set(xlabel='Weight', ylabel='Parameter')
    sns.despine()
    plot_res(results, X, ax=ax)
    
    pdf.savefig(f)

In [ ]:
# # Prep a list to store results 
# all_res = [] 

# full_data_psych['img_cat'] = full_data_psych.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

# model_df = full_data_psych[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 'condition',
#                            'alpha_pos', 'alpha_neg', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'sds_score', 'depr_state',
#                            'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat']].copy()

# model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# # Make a new response variable, memory outcome 
# model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
#                                        'false_alarms']].idxmax(1).to_frame('memory_outcome')

# print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# # CODE THE VARIABLES: 
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# # Subject level traits need to be filled across NaNs
# model_df['alpha_pos'] = model_df.groupby('participant')['alpha_pos'].ffill()
# model_df['alpha_pos'] = model_df.groupby('participant')['alpha_pos'].bfill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].ffill()
# model_df['alpha_neg'] = model_df.groupby('participant')['alpha_neg'].bfill()
# model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
# model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# cat_type = CategoricalDtype(categories=['low', 'med', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)
      
# # Define the dependent variable and the independent variables
# dependent_var = 'memory_outcome'
# independent_vars = ['rpe', 'pm']

# # Set response variable as int
# model_df[dependent_var] = model_df[dependent_var].astype(int)
# # Zscore independent variables: 
# for factor in independent_vars: 
#     model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
#     # Set nan's to 0 to capture the LACK of RPE 
#     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)
      
# model_df = model_df[model_df.condition=='day_2']    
      
# model_df = model_df[model_df.depr_state!='med']
      
# cat_type = CategoricalDtype(categories=['low', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)
      

In [ ]:
# y = 'memory_outcome'
# X = ['zrpe', 'zpm', 'depr_state']
# within_subject_X = ['zrpe', 'zpm']
# Intx = ['C(depr_state):zrpe', 'C(depr_state):zpm']
# rand_effect = ['participant']
# categorical = ['depr_state']

# # Drop nan data
# model_df = model_df.dropna(subset=X)

# label = (f"{y}" + "_{}"*len(X)).format(*X) + '_experiment2_day2'

# model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
#                            priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
#                            target_accept=0.9, model_fam='bernoulli', 
#                            output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
#                            return_model=True,
#                            categorical_baseline=True, 
#                            save_model_res = True, 
#                            label=label)


# f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
# ax.set(xlabel='Weight', ylabel='Parameter')
# sns.despine()
# plot_res(results, X, ax=ax)

In [ ]:
# # Prep a list to store results 
# all_res = [] 

# full_data_psych['img_cat'] = full_data_psych.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

# model_df = full_data_psych[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
#                            'alpha_pos', 'alpha_neg', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'conf_choice',
#                            'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge', 'condition', 'img_cat', 'sds_score',
#                            'depr_state', 'anx_s_state', 'anx_t_state', 'ocd_state']].copy()

# model_df.rename(columns={'image_dprime':'pm'}, inplace=True)

# model_df[['hits', 'misses', 'false_alarms', 'correct_rejections']] = model_df[['hits', 'misses', 'false_alarms', 'correct_rejections']].astype(int)

# cat_type = CategoricalDtype(categories=['day_1', 'day_2'], ordered=True)
# model_df.condition = model_df.condition.astype(cat_type)


# cat_type = CategoricalDtype(categories=['low', 'med', 'high'], ordered=True)
# model_df.depr_state = model_df.depr_state.astype(cat_type)


# # Make a new response variable, memory outcome 
# model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
#                                        'false_alarms']].idxmax(1).to_frame('memory_outcome')

# # CODE THE VARIABLES: 
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
# model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1

# model_df['memory_outcome'] = model_df['memory_outcome'].astype(int)
# model_df = model_df[model_df.condition=='day_1']

# # Define the dependent variable and the independent variables
# dependent_var = 'memory_outcome'
# independent_vars = ['rpe', 'pm']

# # Set response variable as in 
# model_df[dependent_var] = model_df[dependent_var].astype(int)
# # Zscore independent variables: 
# for factor in independent_vars: 
#     model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
#     # Set nan's to 0 to capture the LACK of RPE 
#     model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

# independent_vars = ['zrpe', 'zpm']

In [ ]:
# y = 'memory_outcome'
# X = ['zrpe', 'zpm', 'depr_state']
# Intx = ['depr_state:zrpe', 'depr_state:zpm']
# rand_effect = ['participant']
# categorical = ['depr_state']

# # Drop nan data
# model_df = model_df.dropna(subset=X)

# model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=False, 
#                            priors=None, categorical=categorical, cores=None, chains=3, tune=1500, draws=1500, 
#                            target_accept=0.8, model_fam='bernoulli', 
#                            output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
#                            return_model=True,
#                            categorical_baseline=True, 
#                            save_model_res = True, 
#                            label=None)



In [ ]:
y = 'memory_outcome'
X = ['zrpe', 'zpm', 'zsds_score', 'zalpha', 'zbeta']

label = (f"{y}" + "_{}"*len(X)).format(*X) + '_experiment2_day1'

result_csv = pd.read_csv(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_summary.csv')
result_csv.rename(columns={'Unnamed: 0': 'parameter'}, inplace=True)
# # pattern = r'\[(.*?)\]'
pattern = r'\[(\d.*?)]'

# use the re.search() function to search for the first occurrence of the pattern
# match = re.search(pattern, input_string)
result_csv['participant'] = result_csv.parameter.str.extract(pattern)

pm_index = result_csv[result_csv.parameter.str.contains('zpm')].dropna(subset=['participant'])['parameter'].index 
result_csv.loc[pm_index, 'parameter'] = 'zpm_weight'
rpe_index = result_csv[result_csv.parameter.str.contains('zrpe')].dropna(subset=['participant'])['parameter'].index 
result_csv.loc[rpe_index, 'parameter'] = 'zrpe_weight'

In [ ]:
result_csv.participant.unique().shape

In [ ]:
# Prep a list to store results 
all_res = [] 

full_data_psych['img_cat'] = full_data_psych.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

model_df = full_data_psych[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                           'alpha', 'beta', 'Sex', 'reward', 'participant', 'hit_rate', 'conf_choice',
                           'false_alarm_rate', 'gamble_rt_log', 'recog_rt_log', '_merge', 'condition', 'img_cat',
                           'depr_state', 'anx_s_state', 'anx_t_state', 'ocd_state']].copy()

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)


# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1

model_df = model_df[model_df.condition=='day_2']

# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['rpe', 'pm']

# Set response variable as in 
model_df[dependent_var] = model_df[dependent_var].astype(int)
# Zscore independent variables: 
for factor in independent_vars: 
    model_df[f'z{factor}'] = zscore(model_df[factor], nan_policy='omit').values 
    # Set nan's to 0 to capture the LACK of RPE 
    model_df[f'z{factor}'] = model_df[f'z{factor}'].fillna(0)

independent_vars = ['zrpe', 'zpm']

all_participants = model_df.participant.unique()
model_betas = [] 

for pid in all_participants:
    dfs = [] 
    models = {} 
    subj_df = model_df[model_df.participant==pid]
    
    # make regression weights based on response var frequency
    y_ind = subj_df.memory_outcome.astype(int)
    recip_freq = 1. / np.bincount(y_ind)
    recip_freq /= np.mean(recip_freq)
    wts = recip_freq[y_ind]
    
    if (y_ind.sum()<10) | (y_ind.sum()>subj_df.shape[0]-10):
        print(subj_df.dprime.mean(), y_ind.sum())
        print('model failure - imbalanced data')
        continue
    else:
        # Make the model formula
        formula = f'{dependent_var} ~ 1+'+'+'.join(independent_vars)
        # Make the model
        model = smf.glm(formula = formula, family=sm.families.Binomial(), data=subj_df, freq_weights=wts)
        # Set the baseline 
        model._ynames_map = {0: 0, 1: 1}
        # Fit the model 
        try:
            results = model.fit()
        except: 
            print('model failure - perfect separability')
            continue

    # Collect the individual subject model results. 
    glm_df_temp = results.params.to_dict()
    glm_df = pd.DataFrame(columns=['subj_dprime', 'participant', 'alpha_pos', 'alpha_neg', 'beta'])
    glm_df['GLMBeta'] = list(glm_df_temp.values())
    glm_df['GLMFactor'] = list(glm_df_temp.keys())
    glm_df['subj_dprime'] = subj_df.dprime.mean()
    glm_df['subj_reward'] = subj_df.reward.sum()
    glm_df['subj_rpe'] = subj_df.rpe.mean() 
    glm_df['subj_HR'] = subj_df.hit_rate.mean()
    glm_df['subj_FAR'] = subj_df.false_alarm_rate.mean()
    glm_df['subj_gamble_rt'] = subj_df.gamble_rt_log.mean()
    glm_df['subj_recog_rt'] = subj_df.recog_rt_log.mean()
    glm_df['alpha'] = subj_df.alpha.mean()
    glm_df['beta'] = subj_df.beta.mean()
    glm_df['participant'] = pid
    glm_df['zrpe_weight'] = glm_df_temp['zrpe']
    glm_df['zpm_weight'] = glm_df_temp['zpm']
    glm_df['zrpe_weight_mm'] = result_csv[(result_csv['participant'].str.contains(pid)) & (result_csv.parameter=='zrpe_weight')]['mean'].values[0]
    glm_df['zpm_weight_mm'] = result_csv[(result_csv['participant'].str.contains(pid)) & (result_csv.parameter=='zpm_weight')]['mean'].values[0]
    glm_df['depr_state'] = subj_df.depr_state.iloc[0]
    glm_df['anx_s_state'] = subj_df.anx_s_state.iloc[0]
    glm_df['anx_t_state'] = subj_df.anx_t_state.iloc[0]
    glm_df['ocd_state'] = subj_df.ocd_state.iloc[0]


    model_betas.append(glm_df)
model1_df = pd.concat(model_betas)
model1_df.reset_index(drop=True, inplace=True)

# # Group the data by participant 
model1_df_subj = model1_df.groupby(['participant']).mean().reset_index()
model1_df_subj = model1_df_subj.merge(model1_df['participant'], on='participant').drop_duplicates()
model1_df_subj = model1_df_subj.merge(full_data_psych[['participant', 
                                                   'anx_t_state', 'stai_t_score',
                                                   'depr_state', 'sds_score',
                                                   'anx_s_state', 'stai_s_score', 'ocirscore',
                                                   'ocd_state']], on='participant').drop_duplicates()




In [ ]:
# f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
# sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='alpha', y='subj_dprime', color = 'gray',
#             ci=None, scatter_kws={'s':6, 'alpha':0.5})
# sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='alpha', y='subj_dprime', color = 'red', 
#             ci=None, scatter_kws={'s':6, 'alpha':0.5})
# sns.despine()
# ax.set(xticks=np.linspace(0,1,5), ylabel="subject d'")
# ax.set_ylabel('subject dprime')

In [ ]:
# f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
# sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='beta', y='subj_dprime', color = 'gray',
#             ci=None, scatter_kws={'s':6, 'alpha':0.5})
# sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='beta', y='subj_dprime', color = 'red', 
#             ci=None, scatter_kws={'s':6, 'alpha':0.5})
# sns.despine()
# # ax.set(xticks=np.linspace(0,1,5), ylabel="subject d'")
# ax.set_ylabel('subject dprime')

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_ Fig_LogRegress_WeightsvMemory_Depr_Day2.pdf') as pdf:    

    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=True)
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='zrpe_weight_mm', y='subj_dprime', ci=None, ax=ax[0], scatter_kws={'alpha':0.33}, color='gray')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='zpm_weight_mm', y='subj_dprime', ci=None, ax=ax[1], scatter_kws={'alpha':0.33}, color='gray')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='zrpe_weight_mm', y='subj_dprime', ci=None, ax=ax[0], scatter_kws={'alpha':0.33}, color='r')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='zpm_weight_mm', y='subj_dprime', ci=None, ax=ax[1], scatter_kws={'alpha':0.33}, color='r')
    ax[0].set(xlabel = 'RPE contribution to memory', ylabel='memory (d)')
    ax[1].set(xlabel = 'PM contribution to memory', ylabel='memory (d)')
    sns.despine()
    f.tight_layout()
    
    pdf.savefig()
    

In [ ]:
r1, p1 = spearmanr(model1_df_subj[model1_df_subj.depr_state=='low'].dropna(subset='subj_dprime').zrpe_weight_mm, 
        model1_df_subj[model1_df_subj.depr_state=='low'].dropna(subset='subj_dprime').subj_dprime)
r2, p2 = spearmanr(model1_df_subj[model1_df_subj.depr_state=='high'].dropna(subset='subj_dprime').zrpe_weight_mm, 
        model1_df_subj[model1_df_subj.depr_state=='high'].dropna(subset='subj_dprime').subj_dprime)
F, p = compare_corr(r1, r2, 
                    model1_df_subj[model1_df_subj.depr_state=='low'].dropna(subset='subj_dprime').subj_dprime.shape[0], 
                    model1_df_subj[model1_df_subj.depr_state=='high'].dropna(subset='subj_dprime').subj_dprime.shape[0])
print(F, p )

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(12,4))
sns.barplot(data=model1_df, x='GLMFactor', y='GLMBeta',  errorbar='se', facecolor=None)
ax.set(xlabel='Model Factor', ylabel='Coefficient')

In [ ]:
f, ax = plt.subplots(1,2, dpi=300, figsize=(4,2))
sns.kdeplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='zrpe_weight', color='gray', ax=ax[0])
sns.kdeplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='zrpe_weight', color='red', ax=ax[0])
sns.kdeplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='zpm_weight', color='gray', ax=ax[1])
sns.kdeplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='zpm_weight', color='red', ax=ax[1])
f.tight_layout()

In [ ]:
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=True)
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='subj_reward', y='zrpe_weight', ci=None, ax=ax[0], scatter_kws={'alpha':0.33}, color='gray')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='subj_reward', y='zrpe_weight', ci=None, ax=ax[0], scatter_kws={'alpha':0.33}, color='red')

    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='subj_reward', y='zpm_weight', ci=None, ax=ax[1], scatter_kws={'alpha':0.33}, color='gray')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='subj_reward', y='zpm_weight', ci=None, ax=ax[1], scatter_kws={'alpha':0.33}, color='red')

    ax[0].set(xlabel = 'reward (points)', ylabel='rpe weight')
    ax[1].set(xlabel = 'reward (points)', ylabel='pm weight')
    sns.despine()

In [ ]:
    f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharey=True)
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='zrpe_weight', y='subj_dprime', robust=True,
                ci=None, ax=ax[0], scatter_kws={'alpha':0.33}, color='gray')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='low'], x='zpm_weight', y='subj_dprime', robust=True,
                ci=None, ax=ax[1], scatter_kws={'alpha':0.33}, color='gray')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='zrpe_weight', y='subj_dprime', robust=True,
                ci=None, ax=ax[0], scatter_kws={'alpha':0.33}, color='red')
    sns.regplot(data=model1_df_subj[model1_df_subj.depr_state=='high'], x='zpm_weight', y='subj_dprime', robust=True,
                ci=None, ax=ax[1], scatter_kws={'alpha':0.33}, color='red')
    
    ax[0].set(xlabel = 'rpm weight', ylabel='memory (d)')
    ax[1].set(xlabel = 'pm weight', ylabel='memory (d)')
    sns.despine()

In [ ]:
r1, p1 = spearmanr(model1_df_subj[model1_df_subj.depr_state=='low'].dropna(subset='subj_dprime').zrpe_weight, 
        model1_df_subj[model1_df_subj.depr_state=='low'].dropna(subset='subj_dprime').subj_dprime)
r2, p2 = spearmanr(model1_df_subj[model1_df_subj.depr_state=='high'].dropna(subset='subj_dprime').zrpe_weight, 
        model1_df_subj[model1_df_subj.depr_state=='high'].dropna(subset='subj_dprime').subj_dprime)
F, p = compare_corr(r1, r2, 
                    model1_df_subj[model1_df_subj.depr_state=='low'].dropna(subset='subj_dprime').subj_dprime.shape[0], 
                    model1_df_subj[model1_df_subj.depr_state=='high'].dropna(subset='subj_dprime').subj_dprime.shape[0])
print(F, p )

In [ ]:
r2